# Qualitative Codebook Builder

Created by [Matt Artz](https://www.mattartz.me/) — Advancing AI Anthropology through computational approaches to qualitative research.

## What This Notebook Does

This notebook transforms academic literature and source documents into structured, research-ready codebooks through AI-assisted analysis. By systematically extracting theoretical constructs, methodological approaches, and emergent concepts from uploaded texts, researchers can rapidly develop coding frameworks that maintain methodological rigor while significantly reducing the time investment traditionally required for codebook development.

The extraction process is grounded in your research question and shaped by your chosen analytical lens—selected from 42 built-in lenses spanning critical, interpretive, feminist, decolonial, STS, and many more traditions—or a custom lens you define. Each lens configures the AI to attend to different dimensions of the data, enabling **epistemic pluralism**: running multiple lenses in parallel on the same documents to generate productively different codebooks from the same source material.

After extraction, codes undergo semantic deduplication, AI-assisted consolidation, and hierarchical grouping to produce a refined, navigable codebook. Each generated code includes definitions, inclusion/exclusion criteria, and contextual examples drawn directly from source materials.

## Key Features

- **Analytical lens system**: Choose from 42 built-in analytical lenses spanning critical, interpretive, feminist, decolonial, STS, and many more traditions—or define your own custom lens
- **Research context grounding**: All extraction is anchored in your project name, research question, and data type
- **Parallel multi-lens generation**: Run multiple lenses concurrently to generate productively different codebooks from the same data
- **Quality validation and distinctness**: Embedding-based semantic deduplication merges codes with similar meanings, alongside automatic validation of definitions, criteria, and examples
- **AI-assisted consolidation and grouping**: A second-pass review identifies overlaps, suggests merges and splits, improves definitions, and organizes codes into navigable groups
- **Batch processing with checkpointing**: Concurrent API calls with automatic checkpoint saves and per-document progress tracking for fault tolerance
- **QDA software export**: ATLAS.ti (Excel codebook), NVivo (REFI-QDA codebook, .qdc), and general-purpose formats (CSV, JSON, Markdown)

## Workflow

1. Install dependencies and configure API credentials
2. Define your research context: project name, research question, and data type
3. Select one or more analytical lenses to guide code extraction
4. Upload source documents (PDF, DOCX, TXT)
5. Execute the extraction pipeline (runs in parallel across lenses if multiple selected) and review the semantic deduplication and consolidation results
6. Export codebook(s) in desired formats—each lens produces its own codebook with lens metadata

## Applications

This notebook supports qualitative research workflows from dissertation fieldwork to applied research projects. It is particularly useful for computational analysis using the tools in my AI Anthropology Toolkit.

## Methodological Positioning

This notebook represents a computational anthropology approach—using AI to enhance rather than replace traditional qualitative methods. The analytical lens system operationalizes the concept of **epistemological prompting** from Multi-Agent Ethnography (Artz, 2026): configuring AI agents to adopt specific analytical orientations that shape what they attend to in data. The extraction process prepares data for human interpretation; it does not perform the interpretive work itself. Researchers maintain full analytical authority over code definitions, relationships, and theoretical significance.

## Target Audience

Designed for qualitative researchers at all levels, from graduate students developing dissertation codebooks to research teams standardizing coding frameworks across multi-site studies.

## Technical Approach

The notebook uses large language models (Anthropic Claude) to perform semantic analysis of document chunks, extracting candidate codes based on the configured analytical lens and research question. Each lens modifies the system prompt to foreground different analytical priorities. Asynchronous batch processing enables efficient handling of large document sets, while quality validation ensures extracted codes meet methodological standards. Post-extraction, sentence-transformer embeddings and an AI consolidation pass refine the raw output into a coherent, hierarchically organized codebook.

## Contributing to AI Anthropology

This notebook is part of an emerging methodological framework that positions AI as a collaborative tool in anthropological research. By developing and sharing computational approaches to traditional qualitative methods, we can explore how machine learning augments ethnographic practice while maintaining the interpretive depth that defines anthropological inquiry.

## AI Anthropology Toolkit

- **[Audio Transcription with Whisper](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Audio_Transcription_Whisper.ipynb)**: Local Whisper transcription of interview recordings with optional speaker diarization
- **[Interview Transcript Semantic Chunker](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Interview_Transcript_Semantic_Chunker.ipynb)**: AI-assisted segmentation of interview transcripts into thematically coherent units
- **[Qualitative Codebook Builder](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Qualitative_Codebook_Builder.ipynb)**: AI-assisted extraction of codes and themes from source documents (this notebook)
- **[Coding and Thematic Analysis](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Coding_and_Thematic_Analysis.ipynb)**: AI-assisted application of codebook frameworks to segmented transcripts

## License & Attribution

This work is licensed under the [PolyForm Noncommercial License 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0): free for noncommercial use — research, education, and nonprofit work — with attribution to Matt Artz appreciated. For commercial licensing, contact [Matt Artz](https://www.mattartz.me/).

## Citation

Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## Setup and Installation

*Install required Python packages and import necessary libraries for semantic analysis, file processing, concurrent execution, and interactive widgets. Run this cell first to ensure all dependencies are available.*

In [ ]:
# Install and import required packages

!pip install -q anthropic pypdf python-docx nltk pandas openpyxl lxml sentence-transformers scipy

import os
import json
import re
import time
import concurrent.futures
import logging
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from collections import Counter
from dataclasses import dataclass, field
import xml.etree.ElementTree as ET
from xml.dom import minidom
import uuid

# Suppress pypdf warnings for cleaner output
logging.getLogger('pypdf').setLevel(logging.ERROR)

import anthropic
import pandas as pd
import numpy as np
from pypdf import PdfReader
from docx import Document
import nltk
from nltk.tokenize import sent_tokenize
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import files
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill

# Sentence transformers for semantic deduplication
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine as cosine_distance

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load sentence transformer model for semantic similarity
print("Loading sentence transformer model for semantic deduplication...")
_embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✓ All packages installed and imported successfully")

## Data Structures and Configuration

*Define the core data structures for code entries and configuration parameters. These classes establish the structure for codebook entries following methodological best practices.*

In [ ]:
# Core data structures and analytical lens definitions (42 lenses)

@dataclass
class CodeEntry:
    """Structure for each code following methodological guidelines"""
    label: str = ""
    definition: str = ""
    inclusion_criteria: List[str] = field(default_factory=list)
    exclusion_criteria: List[str] = field(default_factory=list)
    examples: List[Dict] = field(default_factory=list)
    notes: List[Dict] = field(default_factory=list)
    source_documents: List[str] = field(default_factory=list)
    frequency: int = 0
    code_group: str = ""
    extraction_type: str = ""  # theoretical, methodological, or emergent
    stance: str = ""  # which analytical lens generated this code
    created_date: str = field(default_factory=lambda: datetime.now().isoformat())
    last_modified: str = field(default_factory=lambda: datetime.now().isoformat())
    version: str = "1.0.0"


# ── Analytical Lens Definitions (42 lenses, alphabetical) ──────

STANCE_DEFINITIONS = {
    "affect_theory": {
        "name": "Affect Theory",
        "description": "Pre-cognitive intensities, bodily capacities, and affective atmospheres.",
        "prompt_modifier": """Adopt an AFFECT THEORY analytical lens. Focus on:
- Pre-cognitive intensities, bodily capacities, and affective atmospheres
- How affect circulates between bodies, objects, and environments
- The distinction between affect (pre-personal intensity) and emotion (named feeling)
- Affective labor, contagion, and the politics of feeling
- Codes should capture felt intensities and atmospheric qualities beyond named emotions."""
    },
    "anarchist": {
        "name": "Anarchist / Anti-authoritarian",
        "description": "Mutual aid, horizontal organizing, and resistance to hierarchical authority.",
        "prompt_modifier": """Adopt an ANARCHIST / ANTI-AUTHORITARIAN analytical lens. Focus on:
- Mutual aid, solidarity, and horizontal forms of organizing
- Resistance to hierarchical authority, state power, and institutional coercion
- Prefigurative politics and autonomous community practices
- Direct action, self-governance, and voluntary association
- Codes should illuminate anti-hierarchical practices and grassroots self-organization."""
    },
    "business_organizational": {
        "name": "Business / Organizational",
        "description": "Organizational culture, corporate practices, and workplace dynamics.",
        "prompt_modifier": """Adopt a BUSINESS / ORGANIZATIONAL analytical lens. Focus on:
- Organizational culture, workplace norms, and institutional practices
- Management strategies, corporate decision-making, and innovation processes
- How cultural context shapes business practices and economic behavior
- Stakeholder dynamics, corporate social responsibility, and market logics
- Codes should capture organizational processes and workplace cultural dynamics."""
    },
    "cognitive": {
        "name": "Cognitive",
        "description": "Mental models, categorization, decision-making, and cultural cognition.",
        "prompt_modifier": """Adopt a COGNITIVE analytical lens. Focus on:
- Mental models, schemas, and categorization processes
- How cultural knowledge is organized, stored, and retrieved
- Decision-making heuristics, reasoning patterns, and folk theories
- Distributed cognition and the relationship between culture and thought
- Codes should capture cognitive processes, mental models, and reasoning patterns."""
    },
    "computational_digital": {
        "name": "Computational / Digital",
        "description": "Digital platforms, algorithmic systems, and data-mediated social life.",
        "prompt_modifier": """Adopt a COMPUTATIONAL / DIGITAL analytical lens. Focus on:
- Digital platforms, algorithmic mediation, and data-driven systems
- How technology shapes social interaction, identity, and community
- Digital labor, platform economies, and datafication of everyday life
- Methodological innovation in studying online and hybrid social worlds
- Codes should capture digital mediation, platform dynamics, and human-technology entanglements."""
    },
    "critical": {
        "name": "Critical",
        "description": "Power relations, structural constraints, and ideological formations.",
        "prompt_modifier": """Adopt a CRITICAL analytical lens. Focus on:
- Power dynamics, structural inequalities, and systemic constraints
- Ideological assumptions and hegemonic narratives
- Resistance, agency within constraint, and counter-narratives
- How macro-level forces shape individual experience
- Codes should illuminate domination, marginalization, and structural forces."""
    },
    "critical_medical": {
        "name": "Critical Medical",
        "description": "Biomedical power, medicalization, and the social production of health/illness.",
        "prompt_modifier": """Adopt a CRITICAL MEDICAL analytical lens. Focus on:
- Biomedical power, medicalization, and the social construction of disease categories
- How structural inequalities produce differential health outcomes
- Patient experience within institutional medical systems
- Pharmaceutical politics, clinical authority, and epistemic hierarchies in healthcare
- Codes should illuminate how medical systems produce and reproduce social inequalities."""
    },
    "critical_race": {
        "name": "Critical Race",
        "description": "Racial formations, systemic racism, and intersectional oppression.",
        "prompt_modifier": """Adopt a CRITICAL RACE analytical lens. Focus on:
- Racial formations, systemic racism, and the social construction of race
- Intersections of race with class, gender, sexuality, and other axes of power
- Counter-narratives, racial consciousness, and resistance to white supremacy
- Institutional racism, colorblind ideology, and racial microaggressions
- Codes should center racial dynamics and illuminate how racism operates structurally and interpersonally."""
    },
    "decolonial": {
        "name": "Decolonial",
        "description": "Colonial matrices of power, epistemic violence, and pluriversal alternatives.",
        "prompt_modifier": """Adopt a DECOLONIAL analytical lens. Focus on:
- Colonial matrices of power and the persistence of coloniality in knowledge production
- Epistemic violence, subaltern knowledge, and border thinking
- Pluriversal alternatives to Western-centric universalism
- Land, sovereignty, and the politics of who produces legitimate knowledge
- Codes should foreground resistance to colonial epistemologies and center marginalized knowledge systems."""
    },
    "design_anthropology": {
        "name": "Design Anthropology",
        "description": "Design processes, co-creation, future-making, and material interventions.",
        "prompt_modifier": """Adopt a DESIGN ANTHROPOLOGY analytical lens. Focus on:
- Design processes as cultural practices of future-making and world-shaping
- Co-creation, participatory design, and collaborative prototyping
- How designed objects and systems mediate social relations
- The politics of design decisions and their material consequences
- Codes should capture design practices, material interventions, and speculative futures."""
    },
    "economic_anthropology": {
        "name": "Economic Anthropology",
        "description": "Exchange systems, value creation, labor, and economic pluralism.",
        "prompt_modifier": """Adopt an ECONOMIC ANTHROPOLOGY analytical lens. Focus on:
- Diverse economies, exchange systems, and forms of value creation
- Labor, livelihoods, and the social embeddedness of economic activity
- Gift economies, reciprocity, redistribution, and market logics
- How economic practices are culturally constituted and morally evaluated
- Codes should capture economic practices, value systems, and material livelihood strategies."""
    },
    "environmental_political_ecology": {
        "name": "Environmental / Political Ecology",
        "description": "Human-environment relations, resource politics, and ecological knowledge.",
        "prompt_modifier": """Adopt an ENVIRONMENTAL / POLITICAL ECOLOGY analytical lens. Focus on:
- Human-environment relations and the politics of natural resource access and control
- Environmental knowledge, ecological practices, and landscape management
- Climate change, environmental justice, and uneven ecological vulnerabilities
- How power shapes who benefits from and bears the costs of environmental change
- Codes should capture environmental practices, resource politics, and ecological knowledge systems."""
    },
    "evaluation": {
        "name": "Evaluation",
        "description": "Program effectiveness, practical outcomes, and stakeholder perspectives.",
        "prompt_modifier": """Adopt an EVALUATION analytical lens. Focus on:
- Program effectiveness, implementation fidelity, and practical outcomes
- Stakeholder perspectives, needs assessments, and utilization-focused analysis
- What works, for whom, under what conditions, and why
- Actionable findings, recommendations, and evidence for decision-making
- Codes should capture program processes, outcomes, and stakeholder experiences relevant to improvement."""
    },
    "feminist": {
        "name": "Feminist",
        "description": "Gendered power, patriarchy, intersectionality, and embodied experience.",
        "prompt_modifier": """Adopt a FEMINIST analytical lens. Focus on:
- Gendered power relations, patriarchal structures, and intersecting oppressions
- Embodied experience, reproductive labor, and the politics of care
- How gender norms are produced, performed, and contested
- Feminist epistemology: situated knowledge, standpoint, and reflexivity
- Codes should illuminate gendered dynamics, intersectional power, and women's/marginalized experiences."""
    },
    "hermeneutic": {
        "name": "Hermeneutic",
        "description": "Interpretation, the hermeneutic circle, textual meaning, and understanding.",
        "prompt_modifier": """Adopt a HERMENEUTIC analytical lens. Focus on:
- Processes of interpretation and the hermeneutic circle (part-whole relationships)
- How meaning is constituted through dialogue between text, context, and interpreter
- Pre-understanding, prejudice (in the Gadamerian sense), and horizons of meaning
- The conditions under which understanding becomes possible
- Codes should capture interpretive processes, meaning-making, and the contextual conditions of understanding."""
    },
    "historical_archival": {
        "name": "Historical / Archival",
        "description": "Historical processes, archival sources, memory, and temporal change.",
        "prompt_modifier": """Adopt a HISTORICAL / ARCHIVAL analytical lens. Focus on:
- Historical processes, temporal change, and the longue durée of social phenomena
- Archival sources, documentary evidence, and the politics of the archive
- Collective memory, oral history, and contested narratives of the past
- How present conditions are shaped by historical trajectories and path dependencies
- Codes should capture historical processes, temporal dynamics, and the relationship between past and present."""
    },
    "indigenous_methodologies": {
        "name": "Indigenous Methodologies",
        "description": "Relational accountability, community sovereignty, and Indigenous knowledge systems.",
        "prompt_modifier": """Adopt an INDIGENOUS METHODOLOGIES analytical lens. Focus on:
- Relational accountability, reciprocity, and community-centered research ethics
- Indigenous knowledge systems, oral traditions, and land-based epistemologies
- Sovereignty over knowledge production and community self-determination
- Storytelling as method, ceremony as research practice, and place-based knowing
- Codes should center Indigenous ways of knowing, relational ethics, and community sovereignty over knowledge."""
    },
    "infrastructure_studies": {
        "name": "Infrastructure Studies",
        "description": "Built systems, maintenance, breakdown, and the politics of infrastructure.",
        "prompt_modifier": """Adopt an INFRASTRUCTURE STUDIES analytical lens. Focus on:
- Built systems, technical networks, and their social embedding
- Maintenance, repair, breakdown, and the labor of keeping systems running
- How infrastructure becomes visible only upon failure
- The politics of infrastructure: who is served, who is excluded, and who maintains
- Codes should capture infrastructural arrangements, maintenance practices, and the social life of technical systems."""
    },
    "interpretive": {
        "name": "Interpretive",
        "description": "Meaning-making processes, lived experience, and participant perspectives.",
        "prompt_modifier": """Adopt an INTERPRETIVE analytical lens. Focus on:
- Meaning-making processes and subjective experiences
- How participants interpret and construct their social worlds
- Lived experience, emotions, and sense-making narratives
- Cultural symbols, language use, and interpretive frameworks
- Codes should capture how subjects understand their own experiences."""
    },
    "legal_rights_based": {
        "name": "Legal / Rights-based",
        "description": "Legal frameworks, rights claims, justice, and normative orders.",
        "prompt_modifier": """Adopt a LEGAL / RIGHTS-BASED analytical lens. Focus on:
- Legal frameworks, rights claims, and normative orders
- How law is produced, interpreted, and experienced in everyday life
- Legal pluralism, customary law, and the gaps between law-on-paper and law-in-practice
- Justice, access to legal remedies, and the politics of rights discourse
- Codes should capture legal processes, rights claims, and the lived experience of legal systems."""
    },
    "linguistic": {
        "name": "Linguistic",
        "description": "Language structure, discourse patterns, and communicative practices.",
        "prompt_modifier": """Adopt a LINGUISTIC analytical lens. Focus on:
- Language structure, discourse patterns, and communicative practices
- How language constitutes social reality, identity, and relationships
- Code-switching, register, indexicality, and language ideologies
- The pragmatics of speech: what utterances do in social context
- Codes should capture linguistic patterns, discourse strategies, and the social work of language."""
    },
    "material_culture": {
        "name": "Material Culture / Object-oriented",
        "description": "Objects, things, material agency, and human-object entanglements.",
        "prompt_modifier": """Adopt a MATERIAL CULTURE / OBJECT-ORIENTED analytical lens. Focus on:
- Objects, artifacts, and their social lives and trajectories
- Material agency: how things shape, enable, and constrain human action
- Human-object entanglements, assemblages, and material semiosis
- Craft, making, consumption, and the cultural biography of things
- Codes should capture object-human relationships, material practices, and the agency of things."""
    },
    "medical_health_interpretive": {
        "name": "Medical / Health (interpretive)",
        "description": "Illness experience, healing practices, and health-seeking behavior.",
        "prompt_modifier": """Adopt a MEDICAL / HEALTH (INTERPRETIVE) analytical lens. Focus on:
- Illness experience, suffering narratives, and explanatory models of sickness
- Healing practices, therapeutic pluralism, and health-seeking behavior
- The body as a site of cultural inscription and lived experience
- How individuals and communities make sense of health, illness, and care
- Codes should capture illness experiences, healing narratives, and culturally situated health practices."""
    },
    "migration_mobility": {
        "name": "Migration / Mobility Studies",
        "description": "Movement, displacement, borders, and transnational connections.",
        "prompt_modifier": """Adopt a MIGRATION / MOBILITY STUDIES analytical lens. Focus on:
- Movement, displacement, and the politics of borders and belonging
- Transnational connections, diasporic identities, and mobile livelihoods
- Immigration regimes, documentation status, and bureaucratic encounters
- How mobility and immobility are structured by race, class, gender, and geopolitics
- Codes should capture mobility practices, border experiences, and transnational social fields."""
    },
    "mixed_methods": {
        "name": "Mixed-methods",
        "description": "Integration of qualitative and quantitative approaches for complementary insight.",
        "prompt_modifier": """Adopt a MIXED-METHODS analytical lens. Focus on:
- Phenomena amenable to both qualitative interpretation and quantitative measurement
- Patterns that can be triangulated across data types and methods
- Convergence and divergence between qualitative and quantitative findings
- Practical outcomes, transferability, and generalizable insights
- Codes should be concrete enough for quantification while preserving qualitative richness."""
    },
    "multi_sited": {
        "name": "Multi-sited",
        "description": "Connections across sites, following flows of people, things, and ideas.",
        "prompt_modifier": """Adopt a MULTI-SITED analytical lens. Focus on:
- Connections, flows, and circulations across multiple geographic or social sites
- How phenomena manifest differently across locations while remaining connected
- Following people, things, metaphors, or conflicts across institutional and spatial boundaries
- The construction of the field through tracing relations rather than bounding a single site
- Codes should capture cross-site connections, circulations, and the multi-scalar nature of phenomena."""
    },
    "multispecies": {
        "name": "Multispecies / More-than-human",
        "description": "Interspecies relations, non-human agencies, and ecological entanglements.",
        "prompt_modifier": """Adopt a MULTISPECIES / MORE-THAN-HUMAN analytical lens. Focus on:
- Interspecies relations, symbiosis, and multispecies entanglements
- Non-human agencies, animal perspectives, and more-than-human sociality
- How human worlds are co-constituted with plants, animals, fungi, and microbes
- Ecological intimacies, companion species, and the breakdown of nature/culture divides
- Codes should capture interspecies relationships and extend analytical attention beyond the exclusively human."""
    },
    "narrative_life_history": {
        "name": "Narrative / Life History",
        "description": "Storytelling, biographical narratives, and the temporal structuring of experience.",
        "prompt_modifier": """Adopt a NARRATIVE / LIFE HISTORY analytical lens. Focus on:
- Storytelling practices, narrative structure, and biographical trajectories
- How people construct coherent life stories from fragmented experiences
- Plot, temporality, emplotment, and turning points in personal narratives
- The social and cultural shaping of what stories can be told and how
- Codes should capture narrative structures, life story elements, and the temporal organization of experience."""
    },
    "ontological": {
        "name": "Ontological",
        "description": "Multiple worlds, radical difference, and what exists beyond epistemology.",
        "prompt_modifier": """Adopt an ONTOLOGICAL analytical lens. Focus on:
- Ontological difference: multiple worlds rather than multiple perspectives on one world
- What exists, what kinds of beings populate the world, and how reality is constituted
- Taking seriously non-Western ontologies without reducing them to cultural belief
- The politics of ontology: whose reality counts and how worlds are made and unmade
- Codes should attend to ontological commitments, world-making practices, and radical difference in what is real."""
    },
    "performance_performativity": {
        "name": "Performance / Performativity",
        "description": "Social performances, ritual enactments, and performative constitution of identity.",
        "prompt_modifier": """Adopt a PERFORMANCE / PERFORMATIVITY analytical lens. Focus on:
- Social performances, staged interactions, and ritual enactments
- How identities, genders, and social categories are performatively constituted
- The distinction between theatrical performance and performativity (Butler, Austin)
- Embodied practice, gesture, spectacle, and the aesthetics of everyday interaction
- Codes should capture performative acts, ritual practices, and how social realities are enacted into being."""
    },
    "phenomenological": {
        "name": "Phenomenological",
        "description": "First-person lived experience, structures of consciousness, and embodiment.",
        "prompt_modifier": """Adopt a PHENOMENOLOGICAL analytical lens. Focus on:
- First-person lived experience and the structures of consciousness
- Pre-reflective and embodied dimensions of experience
- Epoché: bracketing assumptions to attend to phenomena as they appear
- Temporality, spatiality, and intersubjective dimensions of experience
- Codes should capture the essential structures of how phenomena are experienced."""
    },
    "political_economy_marxian": {
        "name": "Political Economy / Marxian",
        "description": "Class relations, labor, capital accumulation, and structural exploitation.",
        "prompt_modifier": """Adopt a POLITICAL ECONOMY / MARXIAN analytical lens. Focus on:
- Class relations, labor exploitation, and capital accumulation processes
- How economic structures shape social relations, consciousness, and everyday life
- Commodification, alienation, and the production of surplus value
- The relationship between base and superstructure, ideology, and material conditions
- Codes should illuminate class dynamics, labor processes, and structural economic forces."""
    },
    "postcolonial": {
        "name": "Postcolonial",
        "description": "Colonial legacies, subaltern voices, hybridity, and imperial power.",
        "prompt_modifier": """Adopt a POSTCOLONIAL analytical lens. Focus on:
- Colonial legacies and their ongoing effects on formerly colonized societies
- Subaltern voices, representation, and the question of who speaks for whom
- Hybridity, mimicry, and the ambivalence of colonial encounters
- Imperial knowledge systems and their persistent influence on contemporary institutions
- Codes should illuminate colonial residues, subaltern perspectives, and the enduring asymmetries of imperial power."""
    },
    "practice_theory": {
        "name": "Practice Theory",
        "description": "Habitus, embodied routines, and the reproduction of social structures through practice.",
        "prompt_modifier": """Adopt a PRACTICE THEORY analytical lens. Focus on:
- Embodied routines, habitual dispositions, and practical know-how
- How social structures are reproduced and transformed through everyday practices
- Habitus, fields, and the interplay of structure and agency (Bourdieu)
- Material arrangements, bodily competences, and shared understandings that compose practices
- Codes should capture recurring practices, embodied routines, and how social order is produced through doing."""
    },
    "psychoanalytic": {
        "name": "Psychoanalytic",
        "description": "Unconscious processes, desire, fantasy, and psychic dimensions of culture.",
        "prompt_modifier": """Adopt a PSYCHOANALYTIC analytical lens. Focus on:
- Unconscious processes, repression, and the psychic life of social structures
- Desire, fantasy, identification, and their role in cultural reproduction
- Transference, projection, and affective dynamics in social relationships
- How collective anxieties, traumas, and defenses shape cultural formations
- Codes should capture unconscious dynamics, psychic investments, and the emotional undercurrents of social life."""
    },
    "psychological": {
        "name": "Psychological",
        "description": "Individual experience, mental processes, motivation, and well-being.",
        "prompt_modifier": """Adopt a PSYCHOLOGICAL analytical lens. Focus on:
- Individual experience, mental processes, and psychological well-being
- Motivation, coping strategies, and adaptive behavior
- Self-concept, identity formation, and developmental trajectories
- Cognitive and emotional responses to social and environmental conditions
- Codes should capture individual psychological processes, subjective states, and person-level experiences."""
    },
    "public_engaged": {
        "name": "Public / Engaged",
        "description": "Public impact, community engagement, and research as social intervention.",
        "prompt_modifier": """Adopt a PUBLIC / ENGAGED analytical lens. Focus on:
- Public impact, civic engagement, and the social responsibility of research
- Community partnerships, participatory processes, and collaborative knowledge production
- How research can serve communities and contribute to social change
- Translation of findings for public audiences and policy-relevant insights
- Codes should capture publicly relevant dynamics, community needs, and opportunities for engaged scholarship."""
    },
    "queer_theory": {
        "name": "Queer Theory",
        "description": "Heteronormativity, sexuality, gender fluidity, and the politics of deviance.",
        "prompt_modifier": """Adopt a QUEER THEORY analytical lens. Focus on:
- Heteronormativity, compulsory heterosexuality, and the regulation of desire
- Gender fluidity, non-binary identities, and the instability of sexual categories
- How deviance, transgression, and non-normativity challenge social order
- Queer temporalities, kinship formations, and anti-normative world-making
- Codes should illuminate heteronormative assumptions, gender/sexual fluidity, and queer modes of being."""
    },
    "semiotic": {
        "name": "Semiotic",
        "description": "Sign systems, meaning, codes, and the cultural production of significance.",
        "prompt_modifier": """Adopt a SEMIOTIC analytical lens. Focus on:
- Sign systems, codes, and the cultural production of meaning
- How signs (icons, indexes, symbols) mediate social reality
- Intertextuality, connotation, and the layering of cultural meanings
- Semiotic ideologies: beliefs about how signs work and what they represent
- Codes should capture semiotic processes, sign relationships, and the cultural machinery of meaning-making."""
    },
    "structuralist_post_structuralist": {
        "name": "Structuralist / Post-structuralist",
        "description": "Deep structures, binary oppositions, discourse, and the instability of meaning.",
        "prompt_modifier": """Adopt a STRUCTURALIST / POST-STRUCTURALIST analytical lens. Focus on:
- Deep structures, binary oppositions, and underlying systems of classification
- Discourse, power/knowledge, and how language constitutes social reality (Foucault)
- The instability, deferral, and deconstruction of meaning (Derrida)
- How subjects are constituted through discursive practices and disciplinary regimes
- Codes should capture structural patterns, discursive formations, and the constitutive role of language and classification."""
    },
    "sts_actor_network": {
        "name": "STS / Actor-Network",
        "description": "Heterogeneous networks of human and non-human actants; co-production of knowledge and society.",
        "prompt_modifier": """Adopt an STS / ACTOR-NETWORK THEORY analytical lens. Focus on:
- Heterogeneous networks of human and non-human actants
- Translation, enrollment, and stabilization of actor-networks
- How facts, technologies, and social order are co-produced
- Controversies, black-boxing, and obligatory passage points
- Codes should trace associations between actants without privileging human agency."""
    },
    "visual_sensory": {
        "name": "Visual / Sensory",
        "description": "Visual culture, sensory experience, and non-textual modes of knowing.",
        "prompt_modifier": """Adopt a VISUAL / SENSORY analytical lens. Focus on:
- Visual culture, image-making, and the politics of representation
- Sensory experience: sight, sound, smell, taste, touch as modes of knowing
- How non-textual media (photographs, film, objects, soundscapes) convey meaning
- Embodied perception, synaesthesia, and culturally shaped sensoriums
- Codes should capture visual and sensory dimensions of experience, including non-verbal and non-textual phenomena."""
    },
}


class Config:
    """Configuration following best practices from methodological literature"""
    # API Configuration
    ANTHROPIC_API_KEY = ""
    MODEL = "claude-sonnet-5"  # Default model
    MAX_TOKENS = 4096  # Sufficient for complete JSON responses
    TEMPERATURE = 0.2  # Low temperature for consistent JSON formatting

    # Research Context
    PROJECT_NAME = ""
    RESEARCH_QUESTION = ""
    DATA_TYPE = "mixed"  # interviews, surveys, field_notes, social_media, mixed

    # Analytical Lenses
    SELECTED_STANCES = []  # No default — user must choose
    CUSTOM_STANCES = {}  # User-defined stances: {key: {name, description, prompt_modifier}}

    # Codebook Parameters
    MAX_CODE_LABEL_LENGTH = 25
    MIN_DEFINITION_LENGTH = 20
    MAX_TOTAL_CODES = 50  # Soft cap - keep top N by frequency
    MIN_CODE_FREQUENCY = 1  # Default to keep all codes

    # Semantic Deduplication
    SIMILARITY_THRESHOLD = 0.85  # Cosine similarity threshold for merge candidates
    AUTO_MERGE = True  # Merges above the similarity threshold are applied automatically; set to False to only print suggestions

    # Internal Processing Parameters (not exposed in UI)
    MAX_CODES_PER_CHUNK = 6  # Prevents JSON truncation
    BATCH_SIZE = 5  # Concurrent API calls
    RATE_LIMIT_DELAY = 0.3  # Seconds between batches
    MAX_RETRIES = 2  # Retry failed chunks
    CHECKPOINT_INTERVAL = 25  # Save progress every N chunks

    # Advanced Parameters (exposed in Advanced Settings)
    CHUNK_SIZE = 400  # Words per chunk
    OVERLAP = 50  # Word overlap between chunks

    # Paths
    OUTPUT_PATH = "/content/codebook_outputs/"
    VERSION_PATH = "/content/codebook_versions/"

    # Legacy (kept for backwards compatibility but replaced by stance system)
    PURPOSE = ""
    EXTRACTION_FOCUS = ["theoretical", "methodological", "emergent"]  # Default all selected


# Data type descriptions for prompt injection
DATA_TYPE_DESCRIPTIONS = {
    "interviews": "semi-structured or unstructured interview transcripts",
    "surveys": "survey responses and questionnaire data",
    "field_notes": "ethnographic field notes and observational records",
    "social_media": "social media posts, comments, and online discourse",
    "academic_literature": "academic articles, book chapters, reports, and other scholarly publications",
    "mixed": "mixed-method data from multiple source types"
}


def get_all_stances() -> Dict:
    """Return combined built-in + custom analytical lenses"""
    all_stances = dict(STANCE_DEFINITIONS)
    all_stances.update(Config.CUSTOM_STANCES)
    return all_stances


def api_call_with_backoff(client, max_retries=3, **kwargs):
    """Call the Claude API, retrying failures with exponential backoff."""
    for attempt in range(max_retries):
        try:
            return client.messages.create(**kwargs)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt
            print(f"    API call failed ({type(e).__name__}) — retrying in {wait}s")
            time.sleep(wait)


# Global state
client = None
uploaded_documents = {}

print(f"Data structures defined | {len(STANCE_DEFINITIONS)} analytical lenses loaded")

## Research Context and Configuration

*Define your research project, select analytical lenses, and configure analysis parameters. The research question and lens selection shape how codes are extracted from your documents—different lenses will foreground different dimensions of the same data.*

In [ ]:
# Interactive Configuration System with Research Context and Analytical Lens Selection

def create_configuration_interface():
    """Create interactive configuration interface with research context, analytical lens mega-menu, and API settings"""

    instructions_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>Configure Codebook Development</h3>
    <p><strong>Welcome to the Qualitative Codebook Builder!</strong> Set up your research project, choose your analytical lens(es), and configure extraction parameters.</p>
    <div style='background-color: #A3CEF1; padding: 15px; border-radius: 8px; margin: 15px 0; border-left: 4px solid #6096BA;'>
        <p style='color: #274C77; margin: 0; font-weight: bold;'>How Analytical Lenses Work:</p>
        <p style='color: #274C77; margin: 5px 0 0 0;'>Each analytical lens shapes <em>what the AI attends to</em> in your data.
        Select multiple lenses to generate parallel codebooks from the same documents—each revealing different dimensions of your data.
        You can also define a custom lens with your own analytical priorities.</p>
    </div>
    </div>
    """

    style = {'description_width': '200px'}
    layout = widgets.Layout(width='500px')

    # ── Section 1: Research Context ─────────────────────────────────────

    research_header = widgets.HTML("""
    <h3 style='color: #274C77; border-bottom: 2px solid #6096BA; padding-bottom: 8px;'>
        Research Context
    </h3>
    <p style='color: #6096BA; font-size: 13px; margin-top: 4px;'>
        Your research question and data type are injected into all extraction prompts to ground the analysis.
    </p>
    """)

    project_name_widget = widgets.Text(
        value=Config.PROJECT_NAME,
        placeholder='e.g., "AI Governance in Healthcare" or "Dissertation Chapter 3"',
        description='Project Name:',
        style=style,
        layout=layout
    )

    research_question_widget = widgets.Textarea(
        value=Config.RESEARCH_QUESTION,
        placeholder='e.g., "How do healthcare practitioners negotiate AI-driven clinical decision-making within existing institutional power structures?"',
        description='Research Question:',
        style=style,
        layout=widgets.Layout(width='600px', height='80px')
    )

    data_type_widget = widgets.Dropdown(
        options=[
            ('Interviews (semi-structured/unstructured)', 'interviews'),
            ('Surveys (questionnaires, open-ended responses)', 'surveys'),
            ('Field Notes (ethnographic, observational)', 'field_notes'),
            ('Social Media (posts, comments, online discourse)', 'social_media'),
            ('Academic Literature (articles, chapters, reports)', 'academic_literature'),
            ('Mixed (multiple data types)', 'mixed'),
        ],
        value=Config.DATA_TYPE,
        description='Data Type:',
        style=style,
        layout=layout
    )

    # ── Section 2: Analytical Lens Selection ─────────────────────

    stance_header = widgets.HTML("""
    <h3 style='color: #274C77; border-bottom: 2px solid #6096BA; padding-bottom: 8px;'>
        Analytical Lens Selection
    </h3>
    <p style='color: #6096BA; font-size: 13px; margin-top: 4px;'>
        Select one or more lenses. Multiple lenses will generate separate codebooks in parallel.<br/>
        <em>42 lenses available — use the search box to filter, or browse the alphabetical columns below.</em>
    </p>
    """)

    # Search / filter box
    stance_search = widgets.Text(
        value='',
        placeholder='Type to filter lenses (e.g., "critical", "feminist", "STS")...',
        description='',
        layout=widgets.Layout(width='100%', margin='0 0 10px 0'),
        style={'description_width': '0px'}
    )

    # Build stance checkboxes sorted alphabetically by display name
    sorted_stances = sorted(STANCE_DEFINITIONS.items(), key=lambda x: x[1]['name'].lower())
    stance_widgets = {}
    stance_rows = {}  # key -> HBox row (checkbox) for show/hide filtering

    for key, stance in sorted_stances:
        cb = widgets.Checkbox(
            value=(key in Config.SELECTED_STANCES),
            description=stance['name'],
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='auto', margin='1px 0')
        )
        stance_widgets[key] = cb
        stance_rows[key] = cb

    # Distribute into 3 columns (14 per column for 42 stances)
    col_size = (len(sorted_stances) + 2) // 3
    columns = []
    for i in range(3):
        col_stances = sorted_stances[i * col_size : (i + 1) * col_size]
        col_widgets = [stance_rows[key] for key, _ in col_stances]
        columns.append(widgets.VBox(col_widgets, layout=widgets.Layout(
            min_width='280px', padding='0 10px'
        )))

    stance_grid = widgets.HBox(columns, layout=widgets.Layout(
        gap='10px',
        border='1px solid #E7ECEF',
        padding='12px',
        border_radius='8px',
        max_height='450px',
        overflow_y='auto'
    ))

    # Selected stances summary (dynamic)
    stance_summary = widgets.HTML(value="<span style='color: #6096BA; font-size: 13px;'>No lenses selected</span>")

    def update_stance_summary(*args):
        selected = [STANCE_DEFINITIONS[k]['name'] for k, cb in stance_widgets.items() if cb.value]
        if not selected:
            stance_summary.value = "<span style='color: #6096BA; font-size: 13px;'>No lenses selected</span>"
        elif len(selected) <= 5:
            names = ', '.join(selected)
            stance_summary.value = (
                f"<span style='color: #274C77; font-size: 13px; font-weight: bold;'>"
                f"{len(selected)} lens{'es' if len(selected) > 1 else ''} selected:</span> "
                f"<span style='color: #6096BA; font-size: 13px;'>{names}</span>"
            )
        else:
            first_three = ', '.join(selected[:3])
            stance_summary.value = (
                f"<span style='color: #274C77; font-size: 13px; font-weight: bold;'>"
                f"{len(selected)} lenses selected:</span> "
                f"<span style='color: #6096BA; font-size: 13px;'>{first_three}, "
                f"and {len(selected) - 3} more</span>"
            )

    for cb in stance_widgets.values():
        cb.observe(update_stance_summary, names='value')

    # Search filter callback
    def filter_stances(change):
        query = change['new'].strip().lower()
        for key, stance_def in STANCE_DEFINITIONS.items():
            cb = stance_rows[key]
            if not query:
                cb.layout.display = ''
            elif query in stance_def['name'].lower() or query in stance_def['description'].lower():
                cb.layout.display = ''
            else:
                cb.layout.display = 'none'

    stance_search.observe(filter_stances, names='value')

    # Select all / Clear all buttons
    select_all_btn = widgets.Button(
        description='Select All', icon='check-square',
        layout=widgets.Layout(width='120px', height='28px'),
        style={'button_color': '#E7ECEF'}
    )
    clear_all_btn = widgets.Button(
        description='Clear All', icon='square-o',
        layout=widgets.Layout(width='120px', height='28px'),
        style={'button_color': '#E7ECEF'}
    )

    def select_all_stances(b):
        for cb in stance_widgets.values():
            cb.value = True
        update_stance_summary()

    def clear_all_stances(b):
        for cb in stance_widgets.values():
            cb.value = False
        update_stance_summary()

    select_all_btn.on_click(select_all_stances)
    clear_all_btn.on_click(clear_all_stances)

    stance_toolbar = widgets.HBox(
        [stance_search, select_all_btn, clear_all_btn],
        layout=widgets.Layout(gap='8px', align_items='center')
    )

    # Trigger initial summary
    update_stance_summary()

    # Custom stance section
    custom_stance_header = widgets.HTML("""
    <div style='margin-top: 15px; padding-top: 10px; border-top: 1px solid #E7ECEF;'>
        <strong style='color: #274C77;'>Custom Lens (optional)</strong>
        <p style='color: #6096BA; font-size: 12px; margin-top: 2px;'>
            Define your own analytical lens if none of the 42 built-in lenses fit your needs.
        </p>
    </div>
    """)

    custom_stance_name = widgets.Text(
        value='',
        placeholder='e.g., "Disability Studies", "Techno-feminism"',
        description='Lens Name:',
        style=style,
        layout=layout
    )

    custom_stance_description = widgets.Textarea(
        value='',
        placeholder='Describe what this lens emphasizes and what it attends to in data...',
        description='Description:',
        style=style,
        layout=widgets.Layout(width='600px', height='60px')
    )

    custom_stance_prompt = widgets.Textarea(
        value='',
        placeholder='(Optional) Specific prompt guidance for the AI. If left blank, a prompt will be generated from the description above.',
        description='Prompt Guidance:',
        style=style,
        layout=widgets.Layout(width='600px', height='80px')
    )

    custom_stance_active = widgets.Checkbox(
        value=False,
        description='Include this custom lens in extraction',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='auto')
    )

    # ── Section 3: API and Processing Configuration ────────────────────

    api_header = widgets.HTML("""
    <h3 style='color: #274C77; border-bottom: 2px solid #6096BA; padding-bottom: 8px;'>
        API and Processing Configuration
    </h3>
    """)

    api_key_widget = widgets.Password(
        value=Config.ANTHROPIC_API_KEY,
        placeholder='Enter your Anthropic API key',
        description='Anthropic API Key:',
        style=style,
        layout=layout
    )

    model_widget = widgets.Dropdown(
        options=[
            ('Sonnet 5 (Recommended)', 'claude-sonnet-5'),
            ('Haiku 4.5 (Fast & Affordable)', 'claude-haiku-4-5'),
            ('Opus 4.8 (Highest Quality)', 'claude-opus-4-8'),
        ],
        value=Config.MODEL,
        description='Claude Model:',
        style=style,
        layout=layout
    )

    # ── Section 4: Codebook Parameters ─────────────────────────────────

    codebook_header = widgets.HTML("""
    <h3 style='color: #274C77; border-bottom: 2px solid #6096BA; padding-bottom: 8px;'>
        Codebook Parameters
    </h3>
    """)

    max_codes_widget = widgets.IntSlider(
        value=Config.MAX_TOTAL_CODES,
        min=10, max=100, step=5,
        description='Max Total Codes:',
        style=style, layout=layout
    )
    max_codes_help = widgets.HTML(
        "<span style='color: #6096BA; font-size: 12px; margin-left: 25px;'>After extraction, keeps the top N codes by frequency (per lens)</span>"
    )

    min_frequency_widget = widgets.IntSlider(
        value=Config.MIN_CODE_FREQUENCY,
        min=1, max=5, step=1,
        description='Min Occurrences:',
        style=style, layout=layout
    )

    chunk_size_widget = widgets.IntSlider(
        value=Config.CHUNK_SIZE,
        min=200, max=600, step=50,
        description='Chunk Size (words):',
        style=style, layout=layout
    )

    similarity_threshold_widget = widgets.FloatSlider(
        value=Config.SIMILARITY_THRESHOLD,
        min=0.5, max=0.95, step=0.05,
        description='Similarity Threshold:',
        style=style, layout=layout,
        readout_format='.2f'
    )
    similarity_help = widgets.HTML(
        "<span style='color: #6096BA; font-size: 12px; margin-left: 25px;'>Codes with definition similarity above this threshold are merged automatically (lower = more aggressive merging); set Config.AUTO_MERGE = False to only print suggested merges</span>"
    )

    # Extraction focus (kept but now secondary to lens)
    extraction_focus_header = widgets.HTML("""
    <div style='margin-top: 10px;'>
        <strong style='color: #274C77;'>Extraction Focus</strong>
        <span style='color: #6096BA; font-size: 12px;'> — Types of codes to extract</span>
    </div>
    """)

    theoretical_cb = widgets.Checkbox(value='theoretical' in Config.EXTRACTION_FOCUS,
                                      description='Theoretical Constructs', style={'description_width': 'initial'})
    methodological_cb = widgets.Checkbox(value='methodological' in Config.EXTRACTION_FOCUS,
                                         description='Methodological Approaches', style={'description_width': 'initial'})
    emergent_cb = widgets.Checkbox(value='emergent' in Config.EXTRACTION_FOCUS,
                                   description='Emergent Concepts', style={'description_width': 'initial'})

    # ── Buttons ────────────────────────────────────────────────────────

    apply_button = widgets.Button(
        description='Apply Configuration',
        disabled=False, icon='check',
        layout=widgets.Layout(width='200px', height='45px', margin='5px'),
        style={'button_color': '#6096BA', 'font_weight': 'bold'}
    )

    test_api_button = widgets.Button(
        description='Test API Key',
        disabled=False, icon='key',
        layout=widgets.Layout(width='160px', height='45px', margin='5px'),
        style={'button_color': '#A3CEF1', 'font_weight': 'bold'}
    )

    status_output = widgets.Output()

    def apply_configuration(b):
        with status_output:
            status_output.clear_output()

            # Validate research question
            if not research_question_widget.value.strip():
                print("Warning: No research question provided. Extraction will proceed without research grounding.")

            # Build extraction focus
            extraction_focus = []
            if theoretical_cb.value: extraction_focus.append('theoretical')
            if methodological_cb.value: extraction_focus.append('methodological')
            if emergent_cb.value: extraction_focus.append('emergent')
            if not extraction_focus:
                print("Error: Please select at least one extraction focus")
                return

            # Build selected stances from mega-menu
            selected_stances = []
            for key, cb in stance_widgets.items():
                if cb.value:
                    selected_stances.append(key)

            # Handle custom stance
            if custom_stance_active.value and custom_stance_name.value.strip():
                custom_key = re.sub(r'[^a-z0-9_]', '_', custom_stance_name.value.strip().lower())
                custom_desc = custom_stance_description.value.strip()
                custom_prompt = custom_stance_prompt.value.strip()

                if not custom_desc:
                    print("Error: Custom lens needs at least a description")
                    return

                # Auto-generate prompt modifier from description if not provided
                if not custom_prompt:
                    custom_prompt = f"""Adopt a {custom_stance_name.value.strip().upper()} analytical lens. Focus on:
- {custom_desc}
- Codes should reflect the priorities and concerns of this analytical orientation."""

                Config.CUSTOM_STANCES[custom_key] = {
                    "name": custom_stance_name.value.strip(),
                    "description": custom_desc,
                    "prompt_modifier": custom_prompt
                }
                selected_stances.append(custom_key)

            if not selected_stances:
                print("Error: Please select at least one analytical lens")
                return

            try:
                # Apply all settings
                Config.ANTHROPIC_API_KEY = api_key_widget.value
                Config.MODEL = model_widget.value
                Config.PROJECT_NAME = project_name_widget.value.strip()
                Config.RESEARCH_QUESTION = research_question_widget.value.strip()
                Config.DATA_TYPE = data_type_widget.value
                Config.SELECTED_STANCES = selected_stances
                Config.EXTRACTION_FOCUS = extraction_focus
                Config.MAX_TOTAL_CODES = max_codes_widget.value
                Config.MIN_CODE_FREQUENCY = min_frequency_widget.value
                Config.CHUNK_SIZE = chunk_size_widget.value
                Config.SIMILARITY_THRESHOLD = similarity_threshold_widget.value
                Config.PURPOSE = research_question_widget.value.strip()  # backwards compat

                global client
                client = anthropic.Anthropic(api_key=Config.ANTHROPIC_API_KEY)

                os.makedirs(Config.OUTPUT_PATH, exist_ok=True)
                os.makedirs(Config.VERSION_PATH, exist_ok=True)

                # Display summary
                all_stances = get_all_stances()
                stance_names = [all_stances[s]['name'] for s in selected_stances]
                focus_names = {'theoretical': 'Theoretical', 'methodological': 'Methodological', 'emergent': 'Emergent'}

                print("Configuration applied successfully!")
                print(f"")
                print(f"  Project: {Config.PROJECT_NAME or '(not set)'}")
                print(f"  Research Question: {Config.RESEARCH_QUESTION[:120]}{'...' if len(Config.RESEARCH_QUESTION) > 120 else '' if Config.RESEARCH_QUESTION else '(not set)'}")
                print(f"  Data Type: {DATA_TYPE_DESCRIPTIONS.get(Config.DATA_TYPE, Config.DATA_TYPE)}")
                print(f"  Analytical Lenses ({len(stance_names)}): {', '.join(stance_names)}")
                if len(selected_stances) > 1:
                    print(f"  -> {len(selected_stances)} lenses will generate {len(selected_stances)} separate codebooks in parallel")
                    if len(selected_stances) > 3:
                        print(f"     (lenses run up to 3 at a time, with combined API concurrency capped near 9 calls)")
                print(f"  Model: {Config.MODEL}")
                print(f"  Extraction Focus: {', '.join([focus_names[f] for f in extraction_focus])}")
                print(f"  Max Codes: {Config.MAX_TOTAL_CODES} (per lens)")
                print(f"  Similarity Threshold: {Config.SIMILARITY_THRESHOLD:.2f}")

            except Exception as e:
                print(f"Error applying configuration: {e}")

    def test_api_key(b):
        with status_output:
            status_output.clear_output()
            try:
                if not api_key_widget.value:
                    print("Please enter an API key first")
                    return
                print("Testing API connection...")
                test_client = anthropic.Anthropic(api_key=api_key_widget.value)
                response = api_call_with_backoff(
                    test_client, model=model_widget.value, max_tokens=10,
                    messages=[{"role": "user", "content": "Hello"}]
                )
                print(f"API key is valid! Model: {model_widget.value}")
            except Exception as e:
                print(f"API test failed: {e}")

    apply_button.on_click(apply_configuration)
    test_api_button.on_click(test_api_key)

    # ── Layout Assembly ────────────────────────────────────────────────

    # Research context section
    research_section = widgets.VBox([
        research_header,
        project_name_widget,
        research_question_widget,
        data_type_widget,
    ])

    # Analytical lens mega-menu section (full width, spans both columns)
    stance_section = widgets.VBox([
        stance_header,
        stance_toolbar,
        stance_grid,
        stance_summary,
        custom_stance_header,
        custom_stance_active,
        custom_stance_name,
        custom_stance_description,
        custom_stance_prompt,
    ])

    # API section
    api_section = widgets.VBox([
        api_header,
        api_key_widget,
        model_widget,
    ])

    # Codebook params section
    params_section = widgets.VBox([
        codebook_header,
        max_codes_widget, max_codes_help,
        min_frequency_widget,
        chunk_size_widget,
        similarity_threshold_widget, similarity_help,
        extraction_focus_header,
        widgets.HBox([theoretical_cb, methodological_cb, emergent_cb]),
    ])

    buttons_box = widgets.HBox([apply_button, test_api_button])

    # Layout: research + API side by side, then analytical lens mega-menu full width, then params
    top_row = widgets.HBox([research_section, api_section], layout=widgets.Layout(gap='40px'))
    params_row = widgets.VBox([params_section])

    display(HTML(instructions_html))
    display(top_row)
    display(stance_section)
    display(params_row)
    display(buttons_box)
    display(status_output)

create_configuration_interface()

## Document Upload

*Upload your source documents for analysis. Supported formats include PDF, DOCX, and TXT files. Documents are processed and stored for subsequent code extraction.*

In [ ]:
# Document Upload Interface

def extract_text_from_pdf(file_content: bytes) -> str:
    """Extract text from PDF file"""
    import io
    reader = PdfReader(io.BytesIO(file_content))
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

def extract_text_from_docx(file_content: bytes) -> str:
    """Extract text from DOCX file, including table contents"""
    import io
    doc = Document(io.BytesIO(file_content))
    parts = [para.text for para in doc.paragraphs]
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                if cell.text.strip():
                    parts.append(cell.text)
    return "\n".join(parts)

def create_upload_interface():
    """Create document upload interface"""
    global uploaded_documents

    instructions_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>📄 Upload Source Documents</h3>
    <p>Upload the documents you want to analyze. Supported formats: <strong>PDF, DOCX, TXT</strong></p>
    </div>
    """

    upload_button = widgets.Button(
        description='📁 Upload Documents',
        disabled=False,
        tooltip='Click to select and upload your source documents',
        icon='upload',
        layout=widgets.Layout(width='200px', height='50px'),
        style={'button_color': '#6096BA', 'font_weight': 'bold'}
    )

    clear_button = widgets.Button(
        description='🗑️ Clear All',
        disabled=False,
        tooltip='Remove all uploaded documents',
        icon='trash',
        layout=widgets.Layout(width='150px', height='40px'),
        style={'button_color': '#8B8C89', 'font_weight': 'bold'}
    )

    status_output = widgets.Output()

    def handle_upload(b):
        with status_output:
            status_output.clear_output()
            print("📤 Select files to upload...")

        try:
            uploaded = files.upload()

            with status_output:
                for filename, content in uploaded.items():
                    try:
                        if filename.lower().endswith('.pdf'):
                            text = extract_text_from_pdf(content)
                        elif filename.lower().endswith('.docx'):
                            text = extract_text_from_docx(content)
                        elif filename.lower().endswith('.txt'):
                            try:
                                text = content.decode('utf-8')
                            except UnicodeDecodeError:
                                text = content.decode('latin-1')
                                print(f"⚠️ {filename}: not valid UTF-8 — decoded as Latin-1")
                        else:
                            print(f"⚠️ Unsupported format: {filename}")
                            continue

                        uploaded_documents[filename] = text
                        word_count = len(text.split())
                        print(f"✅ {filename}: {word_count:,} words")
                        if filename.lower().endswith('.pdf') and word_count == 0:
                            print(f"⚠️ {filename}: no extractable text — this may be a scanned PDF without a text layer")

                    except Exception as e:
                        print(f"❌ Error processing {filename}: {e}")

                print(f"\n📊 Total documents loaded: {len(uploaded_documents)}")

        except Exception as e:
            with status_output:
                print(f"Upload cancelled or failed: {e}")

    def handle_clear(b):
        global uploaded_documents
        uploaded_documents = {}
        with status_output:
            status_output.clear_output()
            print("🗑️ All documents cleared")

    upload_button.on_click(handle_upload)
    clear_button.on_click(handle_clear)

    display(HTML(instructions_html))
    display(widgets.HBox([upload_button, clear_button]))
    display(status_output)

create_upload_interface()

## Text Processing Utilities

*Core text processing functions for chunking documents into manageable segments. Maintains context across chunk boundaries through controlled overlap.*

In [ ]:
# Text processing utilities

def chunk_text(text: str, chunk_size: int = 400, overlap: int = 50) -> List[str]:
    """Split text into overlapping chunks for processing"""
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_size = 0

    for sentence in sentences:
        sentence_size = len(sentence.split())

        if current_size + sentence_size > chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))
            overlap_sentences = int(overlap * len(current_chunk) / current_size) if current_size > 0 else 0
            current_chunk = current_chunk[-overlap_sentences:] if overlap_sentences > 0 else []
            current_size = sum(len(s.split()) for s in current_chunk)

        current_chunk.append(sentence)
        current_size += sentence_size

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


def sanitize_code_label(label: str) -> str:
    """Ensure code label meets requirements: ≤25 chars, alphanumeric with underscores"""
    label = label.replace(' ', '_')
    label = re.sub(r'([a-z])([A-Z])', r'\1_\2', label)
    label = re.sub(r'[^a-zA-Z0-9_]', '_', label)
    label = re.sub(r'_+', '_', label)
    label = label[:Config.MAX_CODE_LABEL_LENGTH]
    label = label.strip('_')
    return label.upper()


print("✓ Text processing utilities loaded")

## Batch Processing Engine

*Lens-aware extraction engine using concurrent API calls. Each extraction is grounded in your research question and shaped by the selected analytical lens. Processes multiple document chunks simultaneously with progress tracking and automatic checkpointing.*

In [ ]:
# Stance-aware batch processing for code extraction with checkpointing

def get_extraction_prompt(stance_key: str, extraction_focus: List[str], max_codes: int) -> str:
    """
    Build the extraction prompt grounded in research question and shaped by analytical lens.
    """
    all_stances = get_all_stances()
    stance = all_stances[stance_key]
    stance_name = stance['name']
    stance_prompt = stance['prompt_modifier']

    # Build focus-specific instructions
    focus_instructions = []
    if 'theoretical' in extraction_focus:
        focus_instructions.append(
            '- **Theoretical Constructs**: Named theories, frameworks, models, conceptual tools. '
            'Mark these with extraction_type: "theoretical"')
    if 'methodological' in extraction_focus:
        focus_instructions.append(
            '- **Methodological Approaches**: Research methods, analytical techniques, study designs. '
            'Mark these with extraction_type: "methodological"')
    if 'emergent' in extraction_focus:
        focus_instructions.append(
            '- **Emergent Concepts**: Recurring themes, patterns, and ideas across the text. '
            'Mark these with extraction_type: "emergent"')
    focus_text = "\n".join(focus_instructions)

    # Build research context block
    research_context = ""
    if Config.RESEARCH_QUESTION:
        research_context += f"Research Question: {Config.RESEARCH_QUESTION}\n"
    if Config.DATA_TYPE:
        data_desc = DATA_TYPE_DESCRIPTIONS.get(Config.DATA_TYPE, Config.DATA_TYPE)
        research_context += f"Data Type: {data_desc}\n"
    if Config.PROJECT_NAME:
        research_context += f"Project: {Config.PROJECT_NAME}\n"

    if research_context:
        research_block = f"""
RESEARCH CONTEXT:
{research_context.strip()}
"""
    else:
        research_block = ""

    return f"""You are a qualitative research assistant working within a {stance_name} analytical framework.
{research_block}
ANALYTICAL LENS:
{stance_prompt}

Extract codes from the text below that are relevant to the research context above, interpreted through the {stance_name} lens.

CODE TYPES TO EXTRACT:
{focus_text}

IMPORTANT: Extract at most {max_codes} of the most significant codes from this text.
Keep responses concise to ensure complete JSON output.

For each code provide:
- label: ≤25 characters, alphanumeric only, use_underscores (e.g., "ACTOR_NETWORK_THEORY")
- definition: One clear sentence (max 30 words), framed from the {stance_name} perspective
- extraction_type: "theoretical", "methodological", or "emergent"
- example: A brief quote from the text (max 50 words)
- inclusion: When to use this code, from a {stance_name} perspective (max 20 words)
- exclusion: When NOT to use this code (max 20 words)
- code_group: Category grouping based on thematic similarity

Text to analyze:
{{text}}

Return ONLY a valid JSON array. No markdown, no explanation:
[{{{{"label": "CODE_NAME", "definition": "...", "extraction_type": "...", "example": "...", "inclusion": "...", "exclusion": "...", "code_group": "..."}}}}]
"""


def parse_json_response(raw_text: str) -> List[Dict]:
    """
    Robustly parse JSON from API response.
    Handles markdown wrapping and attempts to fix common issues.
    """
    cleaned = raw_text.strip()

    # Remove markdown code blocks
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    # Try direct parsing first
    try:
        result = json.loads(cleaned)
        return result if isinstance(result, list) else [result]
    except json.JSONDecodeError:
        pass

    # Try to find and extract just the JSON array
    array_match = re.search(r'\[.*\]', cleaned, re.DOTALL)
    if array_match:
        try:
            result = json.loads(array_match.group())
            return result if isinstance(result, list) else [result]
        except json.JSONDecodeError:
            pass

    # Try to fix truncated JSON by closing open brackets
    open_brackets = cleaned.count('[') - cleaned.count(']')
    open_braces = cleaned.count('{') - cleaned.count('}')

    if open_brackets > 0 or open_braces > 0:
        last_complete = cleaned.rfind('},')
        if last_complete > 0:
            truncated = cleaned[:last_complete+1] + ']'
            try:
                result = json.loads(truncated)
                return result if isinstance(result, list) else [result]
            except json.JSONDecodeError:
                pass

    return []


def process_single_chunk(chunk: str, doc_name: str, chunk_idx: int,
                         prompt_template: str) -> Tuple[int, str, List[Dict], str]:
    """
    Process a single chunk and return extracted codes.
    Returns: (chunk_idx, doc_name, extracted_codes, status)
    """
    try:
        response = api_call_with_backoff(
            client,
            model=Config.MODEL,
            max_tokens=Config.MAX_TOKENS,
            temperature=Config.TEMPERATURE,
            messages=[{
                "role": "user",
                "content": prompt_template.format(text=chunk)
            }],
            timeout=120.0
        )

        raw_text = response.content[0].text
        extracted = parse_json_response(raw_text)

        if extracted:
            return (chunk_idx, doc_name, extracted, "success")
        else:
            return (chunk_idx, doc_name, [], "parse_error")

    except anthropic.RateLimitError:
        return (chunk_idx, doc_name, [], "rate_limit")
    except anthropic.APIConnectionError:
        return (chunk_idx, doc_name, [], "connection_error")
    except Exception as e:
        return (chunk_idx, doc_name, [], f"error: {str(e)[:80]}")


def save_checkpoint(codebook: Dict[str, CodeEntry], processed_chunks: int,
                    total_chunks: int, stance_key: str = "", status: str = "in_progress"):
    """Save current codebook state to checkpoint file"""
    checkpoint_data = {
        'metadata': {
            'saved_at': datetime.now().isoformat(),
            'processed_chunks': processed_chunks,
            'total_chunks': total_chunks,
            'total_codes': len(codebook),
            'stance': stance_key,
            'status': status
        },
        'codes': {}
    }

    for label, code in codebook.items():
        checkpoint_data['codes'][label] = {
            'definition': code.definition,
            'extraction_type': code.extraction_type,
            'code_group': code.code_group,
            'stance': code.stance,
            'inclusion': code.inclusion_criteria,
            'exclusion': code.exclusion_criteria,
            'examples': [ex['text'] for ex in code.examples[:3]],
            'frequency': code.frequency,
            'sources': code.source_documents
        }

    suffix = f"_{stance_key}" if stance_key else ""
    checkpoint_path = f"{Config.OUTPUT_PATH}codebook_in_progress{suffix}.json"
    with open(checkpoint_path, 'w') as f:
        json.dump(checkpoint_data, f, indent=2)


def normalize_extraction_type(value) -> str:
    """Validate extraction_type against the three expected values.

    Case-insensitive with prefix tolerance; defaults to 'emergent'.
    """
    cleaned = str(value or '').strip().lower()
    if not cleaned:
        return 'emergent'
    for valid in ('theoretical', 'methodological', 'emergent'):
        if cleaned.startswith(valid) or valid.startswith(cleaned):
            return valid
    return 'emergent'


def _integrate_extracted_codes(codebook, extraction_log, extracted_codes, chunk_idx, doc_name, stance_key):
    """Integrate extracted codes from a single chunk into the codebook. Thread-safe for codebook updates."""
    for code_data in extracted_codes:
        if not isinstance(code_data, dict):
            continue

        label = sanitize_code_label(code_data.get('label', ''))
        if not label:
            continue

        if label not in codebook:
            code = CodeEntry(
                label=label,
                definition=code_data.get('definition', 'No definition provided'),
                code_group=code_data.get('code_group', 'Uncategorized'),
                extraction_type=normalize_extraction_type(code_data.get('extraction_type')),
                stance=stance_key,
                frequency=1
            )
            code.source_documents.append(doc_name)
            example_text = code_data.get('example', '')
            if example_text:
                code.examples.append({
                    'text': example_text[:300],
                    'source': doc_name,
                    'chunk': chunk_idx
                })
            if code_data.get('inclusion'):
                code.inclusion_criteria.append(code_data['inclusion'])
            if code_data.get('exclusion'):
                code.exclusion_criteria.append(code_data['exclusion'])

            codebook[label] = code
        else:
            codebook[label].frequency += 1
            if doc_name not in codebook[label].source_documents:
                codebook[label].source_documents.append(doc_name)
            example_text = code_data.get('example', '')
            if example_text and len(codebook[label].examples) < 10:
                codebook[label].examples.append({
                    'text': example_text[:300],
                    'source': doc_name,
                    'chunk': chunk_idx
                })
            if code_data.get('inclusion') and code_data['inclusion'] not in codebook[label].inclusion_criteria:
                codebook[label].inclusion_criteria.append(code_data['inclusion'])
            if code_data.get('exclusion') and code_data['exclusion'] not in codebook[label].exclusion_criteria:
                codebook[label].exclusion_criteria.append(code_data['exclusion'])

        extraction_log.append({
            'timestamp': datetime.now().isoformat(),
            'document': doc_name,
            'chunk': chunk_idx,
            'code': label,
            'stance': stance_key,
            'extraction_type': normalize_extraction_type(code_data.get('extraction_type'))
        })


def extract_codes_batch(documents: Dict[str, str],
                        stance_key: str,
                        extraction_focus: List[str]) -> Tuple[Dict[str, CodeEntry], List]:
    """
    Extract codes using parallel API calls, with worker pools sized so that
    combined in-flight requests across concurrently processed lenses stay near 9.
    All extraction is shaped by the specified analytical lens.
    """
    all_stances = get_all_stances()
    stance_name = all_stances[stance_key]['name']

    codebook = {}
    extraction_log = []
    prompt_template = get_extraction_prompt(stance_key, extraction_focus, Config.MAX_CODES_PER_CHUNK)

    # Prepare all chunks
    all_tasks = []
    doc_chunk_counts = {}
    for doc_name, content in documents.items():
        chunks = chunk_text(content, Config.CHUNK_SIZE, Config.OVERLAP)
        doc_chunk_counts[doc_name] = len(chunks)
        for i, chunk in enumerate(chunks):
            all_tasks.append((chunk, doc_name, i, len(chunks)))

    total_chunks = len(all_tasks)
    focus_names = {'theoretical': 'Theoretical', 'methodological': 'Methodological', 'emergent': 'Emergent'}
    focus_display = ', '.join([focus_names[f] for f in extraction_focus])

    # Cap combined in-flight API calls near 9: up to 3 lenses run concurrently,
    # each with a chunk pool sized so the total stays within the cap
    outer_workers = min(3, max(1, len(Config.SELECTED_STANCES)))
    max_workers = min(max(2, 9 // outer_workers), max(1, total_chunks))

    print(f"\n  [{stance_name}] Starting extraction...")
    print(f"  [{stance_name}] Chunks: {total_chunks} | Focus: {focus_display} | Workers: {max_workers}")

    processed = 0
    failed_tasks = []
    status_counts = Counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(process_single_chunk, chunk, doc_name, idx, prompt_template): (chunk, doc_name, idx, total)
            for chunk, doc_name, idx, total in all_tasks
        }

        for future in concurrent.futures.as_completed(futures):
            chunk_idx, doc_name, extracted_codes, status = future.result()
            status_counts[status] += 1

            if extracted_codes:
                _integrate_extracted_codes(codebook, extraction_log, extracted_codes, chunk_idx, doc_name, stance_key)
            else:
                task_data = futures[future]
                failed_tasks.append(task_data)

            processed += 1

            # Progress every 10 chunks or at the end
            if processed % 10 == 0 or processed == total_chunks:
                pct = (processed / total_chunks) * 100
                print(f"  [{stance_name}] Progress: {processed}/{total_chunks} ({pct:.0f}%) | {len(codebook)} codes")

            # Checkpoint periodically
            if processed % Config.CHECKPOINT_INTERVAL == 0 and processed > 0:
                save_checkpoint(codebook, processed, total_chunks, stance_key)

    # Retry all failed chunks in parallel, up to Config.MAX_RETRIES passes
    for retry_round in range(1, Config.MAX_RETRIES + 1):
        if not failed_tasks:
            break
        print(f"  [{stance_name}] Retry pass {retry_round}/{Config.MAX_RETRIES}: {len(failed_tasks)} failed chunks...")
        time.sleep(1)
        retry_successful = 0
        still_failed = []

        retry_workers = min(max_workers, len(failed_tasks))
        with concurrent.futures.ThreadPoolExecutor(max_workers=retry_workers) as executor:
            retry_futures = {
                executor.submit(process_single_chunk, chunk, doc_name, idx, prompt_template): (chunk, doc_name, idx, total)
                for chunk, doc_name, idx, total in failed_tasks
            }

            for future in concurrent.futures.as_completed(retry_futures):
                chunk_idx, doc_name, extracted_codes, status = future.result()
                if extracted_codes:
                    retry_successful += 1
                    _integrate_extracted_codes(codebook, extraction_log, extracted_codes, chunk_idx, doc_name, stance_key)
                else:
                    still_failed.append(retry_futures[future])

        print(f"  [{stance_name}] Retry pass {retry_round} recovered: {retry_successful} chunks")
        failed_tasks = still_failed

    if failed_tasks:
        failed_by_doc = Counter(doc_name for _, doc_name, _, _ in failed_tasks)
        print(f"  [{stance_name}] {len(failed_tasks)} chunks could not be processed after {Config.MAX_RETRIES} retry passes:")
        for doc_name, count in sorted(failed_by_doc.items()):
            print(f"    {doc_name}: {count} chunks")

    save_checkpoint(codebook, total_chunks, total_chunks, stance_key, status='complete')

    type_counts = Counter(code.extraction_type for code in codebook.values())
    print(f"  [{stance_name}] Extraction complete: {len(codebook)} codes | By type: {dict(type_counts)}")

    return codebook, extraction_log


print("✓ Batch processing engine loaded (concurrent workers with exponential backoff and retry passes)")

## Code Refinement and Validation

*Refine extracted codes through semantic deduplication, AI-assisted consolidation, definition synthesis, example diversity selection, and hierarchical grouping. This multi-step refinement process transforms raw extracted codes into a coherent, navigable codebook.*

In [ ]:
# Code refinement: semantic dedup, AI consolidation, definition synthesis, hierarchical grouping

def compute_code_embeddings(codebook: Dict[str, CodeEntry]) -> Dict[str, np.ndarray]:
    """Compute embeddings for each code's definition for semantic similarity comparison."""
    labels = list(codebook.keys())
    definitions = [codebook[l].definition for l in labels]
    embeddings = _embedding_model.encode(definitions, show_progress_bar=False)
    return {label: emb for label, emb in zip(labels, embeddings)}


def find_semantic_merge_candidates(codebook: Dict[str, CodeEntry],
                                   threshold: float) -> List[Tuple[str, str, float]]:
    """
    Find pairs of codes with semantically similar definitions.
    Returns list of (label_a, label_b, similarity_score) tuples above threshold.
    """
    embeddings = compute_code_embeddings(codebook)
    labels = list(embeddings.keys())
    candidates = []

    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            sim = 1.0 - cosine_distance(embeddings[labels[i]], embeddings[labels[j]])
            if sim >= threshold:
                candidates.append((labels[i], labels[j], sim))

    candidates.sort(key=lambda x: x[2], reverse=True)
    return candidates


def merge_codes(codebook: Dict[str, CodeEntry], label_keep: str, label_remove: str) -> Dict[str, CodeEntry]:
    """Merge label_remove into label_keep, combining their metadata."""
    keep = codebook[label_keep]
    remove = codebook[label_remove]

    # Combine frequencies
    keep.frequency += remove.frequency

    # Combine source documents
    for doc in remove.source_documents:
        if doc not in keep.source_documents:
            keep.source_documents.append(doc)

    # Combine examples (up to 10 total)
    for ex in remove.examples:
        if len(keep.examples) < 10:
            keep.examples.append(ex)

    # Combine inclusion/exclusion criteria
    for inc in remove.inclusion_criteria:
        if inc not in keep.inclusion_criteria:
            keep.inclusion_criteria.append(inc)
    for exc in remove.exclusion_criteria:
        if exc not in keep.exclusion_criteria:
            keep.exclusion_criteria.append(exc)

    del codebook[label_remove]
    return codebook


def semantic_deduplication(codebook: Dict[str, CodeEntry], threshold: float) -> Dict[str, CodeEntry]:
    """
    Find semantically similar codes. When Config.AUTO_MERGE is True (default),
    pairs above the threshold are merged, keeping the higher-frequency code.
    When False, suggested merges are printed without being applied.
    """
    print(f"\n   Step 2: Semantic Deduplication (threshold={threshold:.2f})")
    initial_count = len(codebook)

    candidates = find_semantic_merge_candidates(codebook, threshold)

    if not candidates:
        print(f"   No semantic duplicates found above threshold")
        return codebook

    if not Config.AUTO_MERGE:
        print(f"   AUTO_MERGE is off — {len(candidates)} suggested merge pairs (not applied):")
        for label_a, label_b, sim in candidates:
            print(f"     '{label_a}' <-> '{label_b}' (similarity: {sim:.3f})")
        return codebook

    print(f"   Found {len(candidates)} merge candidate pairs:")

    merged_count = 0
    already_merged = set()

    for label_a, label_b, sim in candidates:
        # Skip if either code was already merged away
        if label_a in already_merged or label_b in already_merged:
            continue
        if label_a not in codebook or label_b not in codebook:
            continue

        # Keep the one with higher frequency
        if codebook[label_a].frequency >= codebook[label_b].frequency:
            keep, remove = label_a, label_b
        else:
            keep, remove = label_b, label_a

        print(f"     Merging '{remove}' -> '{keep}' (similarity: {sim:.3f})")
        codebook = merge_codes(codebook, keep, remove)
        already_merged.add(remove)
        merged_count += 1

    print(f"   Merged {merged_count} codes ({initial_count} -> {len(codebook)})")
    return codebook


def _synthesize_batch(batch_codes, stance_name):
    """Synthesize definitions for a batch of codes via a single API call."""
    prompt = f"""You are refining code definitions for a {stance_name} qualitative codebook.

For each code below, synthesize the best possible definition by considering:
- The current definition
- The example passages where this code was found
- The {stance_name} analytical lens perspective

Return a JSON array with objects containing "label" and "definition" (max 30 words each).
Only improve definitions that are vague or could better reflect the {stance_name} lens.
Keep definitions that are already good.

Codes to refine:
{json.dumps(batch_codes, indent=2)}

Return ONLY a valid JSON array:
[{{"label": "CODE_NAME", "definition": "improved definition..."}}]"""

    try:
        response = api_call_with_backoff(
            client,
            model=Config.MODEL,
            max_tokens=Config.MAX_TOKENS,
            temperature=Config.TEMPERATURE,
            messages=[{"role": "user", "content": prompt}],
            timeout=120.0
        )
        return parse_json_response(response.content[0].text)
    except Exception as e:
        return []


def synthesize_definitions(codebook: Dict[str, CodeEntry], stance_key: str) -> Dict[str, CodeEntry]:
    """
    Use Claude to synthesize improved definitions for codes that have been merged
    or that appeared across multiple chunks (where later definitions might be better).
    Parallelizes across batches of 8 codes with up to 5 concurrent workers.
    """
    all_stances = get_all_stances()
    stance_name = all_stances[stance_key]['name']

    # Only synthesize for codes with multiple examples (indicates multiple occurrences)
    codes_to_synthesize = {
        label: code for label, code in codebook.items()
        if code.frequency > 1 and len(code.examples) > 1
    }

    if not codes_to_synthesize:
        print(f"   Step 3: Definition Synthesis — No multi-occurrence codes to synthesize")
        return codebook

    print(f"   Step 3: Definition Synthesis ({len(codes_to_synthesize)} codes)")

    # Build code summaries
    all_code_summaries = []
    for label, code in codes_to_synthesize.items():
        example_texts = [ex['text'][:150] for ex in code.examples[:5]]
        all_code_summaries.append({
            'label': label,
            'current_definition': code.definition,
            'examples': example_texts,
            'frequency': code.frequency,
            'inclusion': code.inclusion_criteria[:3],
            'exclusion': code.exclusion_criteria[:3]
        })

    # Split into batches of 8 codes and process in parallel
    batch_size = 8
    batches = [all_code_summaries[i:i+batch_size] for i in range(0, len(all_code_summaries), batch_size)]

    updated = 0
    max_workers = min(5, len(batches))

    if len(batches) > 1:
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [executor.submit(_synthesize_batch, batch, stance_name) for batch in batches]
            for future in concurrent.futures.as_completed(futures):
                refined = future.result()
                for item in refined:
                    if isinstance(item, dict) and item.get('label') and item.get('definition'):
                        label = item['label']
                        if label in codebook and item['definition'] != codebook[label].definition:
                            codebook[label].definition = item['definition']
                            codebook[label].last_modified = datetime.now().isoformat()
                            updated += 1
    else:
        # Single batch — no need for thread pool
        refined = _synthesize_batch(all_code_summaries, stance_name)
        for item in refined:
            if isinstance(item, dict) and item.get('label') and item.get('definition'):
                label = item['label']
                if label in codebook and item['definition'] != codebook[label].definition:
                    codebook[label].definition = item['definition']
                    codebook[label].last_modified = datetime.now().isoformat()
                    updated += 1

    print(f"   Refined {updated} definitions")
    return codebook


def select_diverse_examples(codebook: Dict[str, CodeEntry], max_examples: int = 3) -> Dict[str, CodeEntry]:
    """
    For codes with multiple examples, select the most diverse subset using embedding diversity.
    """
    print(f"   Step 4: Example Diversity Selection")

    for label, code in codebook.items():
        if len(code.examples) <= max_examples:
            continue

        # Embed all examples
        texts = [ex['text'] for ex in code.examples]
        embeddings = _embedding_model.encode(texts, show_progress_bar=False)

        # Greedy farthest-point selection: start with first, then iteratively pick
        # the candidate least similar to anything already selected
        selected_indices = [0]
        remaining = list(range(1, len(embeddings)))

        while len(selected_indices) < max_examples and remaining:
            best_idx = None
            best_max_sim = None

            for idx in remaining:
                max_sim = max(
                    1.0 - cosine_distance(embeddings[idx], embeddings[s])
                    for s in selected_indices
                )
                # Pick the candidate whose highest similarity to the selected set is lowest
                if best_max_sim is None or max_sim < best_max_sim:
                    best_max_sim = max_sim
                    best_idx = idx

            selected_indices.append(best_idx)
            remaining.remove(best_idx)

        code.examples = [code.examples[i] for i in sorted(selected_indices)]

    print(f"   Selected most diverse examples (max {max_examples} per code)")
    return codebook


def _distribute_examples_to_children(parent: CodeEntry, children: List[CodeEntry]):
    """Assign each parent example to the child it is most similar to.

    Similarity is computed between the example text and each child's
    label plus definition using the sentence-transformer embeddings.
    Falls back to giving every child all parent examples if assignment fails.
    """
    if not parent.examples or not children:
        return
    try:
        child_texts = [f"{child.label.replace('_', ' ')}: {child.definition}" for child in children]
        child_embeddings = _embedding_model.encode(child_texts, show_progress_bar=False)
        example_texts = [ex['text'] for ex in parent.examples]
        example_embeddings = _embedding_model.encode(example_texts, show_progress_bar=False)
        for ex, ex_emb in zip(parent.examples, example_embeddings):
            sims = [1.0 - cosine_distance(ex_emb, c_emb) for c_emb in child_embeddings]
            children[int(np.argmax(sims))].examples.append(ex)
    except Exception:
        for child in children:
            child.examples = [dict(ex) for ex in parent.examples]


def ai_consolidation_pass(codebook: Dict[str, CodeEntry], stance_key: str) -> Dict[str, CodeEntry]:
    """
    Send the complete code list to Claude for a consolidation review:
    - Identify remaining overlapping or redundant codes
    - Suggest merges with rationale
    - Suggest splits for overly broad codes
    - Improve definitions that are vague or incomplete
    - Recommend hierarchical grouping structure
    """
    all_stances = get_all_stances()
    stance_name = all_stances[stance_key]['name']

    print(f"\n   Step 5: AI-Assisted Consolidation Pass ({stance_name} lens)")

    # Build compact code list for the prompt
    code_summary = []
    for label, code in sorted(codebook.items(), key=lambda x: x[1].frequency, reverse=True):
        code_summary.append({
            'label': label,
            'definition': code.definition,
            'code_group': code.code_group,
            'extraction_type': code.extraction_type,
            'frequency': code.frequency
        })

    research_context = ""
    if Config.RESEARCH_QUESTION:
        research_context = f"\nResearch Question: {Config.RESEARCH_QUESTION}"

    prompt = f"""You are reviewing a qualitative codebook generated from a {stance_name} analytical lens.{research_context}

Review the codebook below and provide consolidation recommendations:

1. MERGE: Identify codes that substantially overlap and should be merged. For each, specify which code to keep and which to absorb.
2. SPLIT: Identify codes that are too broad and should be split into more specific codes.
3. REGROUP: Suggest improved code_group assignments to create a coherent hierarchical structure.
4. IMPROVE: For any codes with vague or incomplete definitions, provide improved versions.

Current codebook ({len(code_summary)} codes):
{json.dumps(code_summary, indent=2)}

Return ONLY a valid JSON object with this structure:
{{
  "merges": [{{"keep": "LABEL_A", "absorb": "LABEL_B", "rationale": "..."}}],
  "splits": [{{"label": "BROAD_CODE", "into": [{{"label": "NEW_1", "definition": "..."}}, {{"label": "NEW_2", "definition": "..."}}], "rationale": "..."}}],
  "regroups": [{{"label": "CODE", "new_group": "Group Name"}}],
  "improved_definitions": [{{"label": "CODE", "definition": "improved definition"}}]
}}

Be conservative — only recommend changes that meaningfully improve the codebook.
If the codebook is already well-structured, return empty arrays."""

    try:
        response = api_call_with_backoff(
            client,
            model=Config.MODEL,
            max_tokens=Config.MAX_TOKENS,
            temperature=Config.TEMPERATURE,
            messages=[{"role": "user", "content": prompt}],
            timeout=180.0
        )

        # Parse with the hardened parser so truncated responses degrade gracefully
        parsed = parse_json_response(response.content[0].text)
        if not parsed or not isinstance(parsed[0], dict):
            print(f"   Consolidation response could not be parsed (possibly truncated) — no changes applied")
            return codebook

        recommendations = parsed[0]

        # Apply merges
        merges = recommendations.get('merges', [])
        merge_count = 0
        for merge in merges:
            keep = merge.get('keep', '')
            absorb = merge.get('absorb', '')
            if keep in codebook and absorb in codebook:
                print(f"     Merge: '{absorb}' -> '{keep}' ({merge.get('rationale', '')})")
                codebook = merge_codes(codebook, keep, absorb)
                merge_count += 1
        if merge_count:
            print(f"   Applied {merge_count} merges")

        # Apply splits
        splits = recommendations.get('splits', [])
        split_count = 0
        for split in splits:
            original_label = split.get('label', '')
            new_codes = split.get('into', [])
            if original_label in codebook and len(new_codes) >= 2:
                original = codebook[original_label]
                created = []
                for new_code_data in new_codes:
                    new_label = sanitize_code_label(new_code_data.get('label', ''))
                    if new_label and new_label not in codebook:
                        new_code = CodeEntry(
                            label=new_label,
                            definition=new_code_data.get('definition', original.definition),
                            code_group=original.code_group,
                            extraction_type=original.extraction_type,
                            stance=original.stance,
                            frequency=max(1, original.frequency // len(new_codes)),
                            source_documents=list(original.source_documents),
                            inclusion_criteria=list(original.inclusion_criteria),
                            exclusion_criteria=list(original.exclusion_criteria)
                        )
                        codebook[new_label] = new_code
                        created.append(new_label)
                if created:
                    _distribute_examples_to_children(original, [codebook[l] for l in created])
                    del codebook[original_label]
                    split_count += 1
                    print(f"     Split: '{original_label}' -> {created}")
        if split_count:
            print(f"   Applied {split_count} splits")

        # Apply regroups
        regroups = recommendations.get('regroups', [])
        regroup_count = 0
        for regroup in regroups:
            label = regroup.get('label', '')
            new_group = regroup.get('new_group', '')
            if label in codebook and new_group:
                codebook[label].code_group = new_group
                regroup_count += 1
        if regroup_count:
            print(f"   Regrouped {regroup_count} codes")

        # Apply improved definitions
        improved = recommendations.get('improved_definitions', [])
        improve_count = 0
        for imp in improved:
            label = imp.get('label', '')
            new_def = imp.get('definition', '')
            if label in codebook and new_def:
                codebook[label].definition = new_def
                codebook[label].last_modified = datetime.now().isoformat()
                improve_count += 1
        if improve_count:
            print(f"   Improved {improve_count} definitions")

        if not (merge_count or split_count or regroup_count or improve_count):
            print(f"   Codebook is well-structured — no changes recommended")

    except Exception as e:
        print(f"   Consolidation pass skipped (error: {str(e)[:60]})")

    return codebook


def refine_and_assess(codebook: Dict[str, CodeEntry],
                      stance_key: str = "") -> Tuple[Dict[str, CodeEntry], Dict]:
    """
    Full refinement pipeline:
    1. Frequency filtering
    2. Semantic deduplication
    3. Definition synthesis (parallel)
    4. Example diversity selection
    5. AI-assisted consolidation pass
    6. Soft cap
    7. Quality validation
    """
    all_stances = get_all_stances()
    stance_name = all_stances.get(stance_key, {}).get('name', 'Unknown') if stance_key else 'Default'

    print(f"\n{'─' * 50}")
    print(f"Refining codebook [{stance_name}]...")
    print(f"{'─' * 50}")
    initial_count = len(codebook)

    # Step 1: Frequency filtering
    codes_below_frequency = [
        label for label, code in codebook.items()
        if code.frequency < Config.MIN_CODE_FREQUENCY
    ]
    for label in codes_below_frequency:
        del codebook[label]
    if codes_below_frequency:
        print(f"   Step 1: Frequency Filtering — removed {len(codes_below_frequency)} codes below threshold ({Config.MIN_CODE_FREQUENCY})")

    # Step 2: Semantic deduplication
    if len(codebook) > 1:
        codebook = semantic_deduplication(codebook, Config.SIMILARITY_THRESHOLD)

    # Step 3: Definition synthesis (parallel batches)
    if stance_key and client:
        codebook = synthesize_definitions(codebook, stance_key)

    # Step 4: Example diversity
    codebook = select_diverse_examples(codebook, max_examples=3)

    # Step 5: AI consolidation pass
    if stance_key and client and len(codebook) > 3:
        codebook = ai_consolidation_pass(codebook, stance_key)

    # Step 6: Soft cap
    if len(codebook) > Config.MAX_TOTAL_CODES:
        sorted_codes = sorted(codebook.items(), key=lambda x: x[1].frequency, reverse=True)
        removed_by_cap = len(codebook) - Config.MAX_TOTAL_CODES
        codebook = dict(sorted_codes[:Config.MAX_TOTAL_CODES])
        print(f"\n   Step 6: Soft Cap — kept top {Config.MAX_TOTAL_CODES} codes by frequency (removed {removed_by_cap})")

    # Step 7: Quality validation
    validation_issues = {
        'missing_definitions': 0,
        'short_definitions': 0,
        'missing_examples': 0
    }
    for label, code in codebook.items():
        if not code.definition or code.definition == 'No definition provided':
            validation_issues['missing_definitions'] += 1
        elif len(code.definition) < Config.MIN_DEFINITION_LENGTH:
            validation_issues['short_definitions'] += 1
        if len(code.examples) < 1:
            validation_issues['missing_examples'] += 1

    total_codes = len(codebook)
    if total_codes > 0:
        issue_count = sum(validation_issues.values())
        # Two issue families are checked per code (definition quality, example presence)
        quality_score = max(0, 1 - (issue_count / (total_codes * 2)))
    else:
        quality_score = 0

    print(f"\n   Step 7: Quality Validation — score {quality_score:.2f}")

    # Count code groups for hierarchy summary
    group_counts = Counter(code.code_group for code in codebook.values())

    assessment = {
        'validation': {
            'quality_score': quality_score,
            'summary': validation_issues,
            'initial_count': initial_count,
            'final_count': len(codebook),
            'stance': stance_key
        },
        'groups': dict(group_counts)
    }

    print(f"\n   Refinement Summary [{stance_name}]:")
    print(f"   Initial codes: {initial_count} -> Final codes: {len(codebook)}")
    print(f"   Quality score: {quality_score:.2f}")
    print(f"   Code groups: {len(group_counts)}")
    for group, count in sorted(group_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"     {group}: {count} codes")

    return codebook, assessment


print("✓ Enhanced refinement functions loaded (parallel definition synthesis, semantic dedup, AI consolidation)")

## Export Functions

*Export codebook in multiple formats optimized for different QDA software platforms. Each lens's codebook is exported separately with lens metadata. Includes proper formats for ATLAS.ti (Excel), NVivo (REFI-QDA codebook, QDC), and general-purpose formats (CSV, JSON, Markdown).*

In [ ]:
# Export functions with lens metadata and per-lens file naming

def _stance_suffix(stance_key: str) -> str:
    """Generate filename suffix from stance key."""
    return f"_{stance_key}" if stance_key else ""


def export_to_csv(codebook: Dict[str, CodeEntry], stance_key: str = ""):
    """Export as general-purpose CSV with stance metadata"""
    all_stances = get_all_stances()
    stance_name = all_stances.get(stance_key, {}).get('name', '') if stance_key else ''
    suffix = _stance_suffix(stance_key)

    rows = []
    for label, code in codebook.items():
        rows.append({
            'code_label': label,
            'definition': code.definition,
            'extraction_type': code.extraction_type,
            'code_group': code.code_group,
            'stance': stance_name,
            'stance_key': stance_key,
            'inclusion_criteria': '; '.join(code.inclusion_criteria),
            'exclusion_criteria': '; '.join(code.exclusion_criteria),
            'example_1': code.examples[0]['text'] if code.examples else '',
            'example_2': code.examples[1]['text'] if len(code.examples) > 1 else '',
            'example_3': code.examples[2]['text'] if len(code.examples) > 2 else '',
            'frequency': code.frequency,
            'source_documents': '; '.join(code.source_documents)
        })

    df = pd.DataFrame(rows)
    df = df.sort_values('frequency', ascending=False)
    filepath = f"{Config.OUTPUT_PATH}codebook{suffix}.csv"
    df.to_csv(filepath, index=False)
    print(f"  codebook{suffix}.csv")


def export_to_json(codebook: Dict[str, CodeEntry], assessment: Dict, stance_key: str = ""):
    """Export as JSON with full stance and research context metadata"""
    all_stances = get_all_stances()
    stance_info = all_stances.get(stance_key, {}) if stance_key else {}
    suffix = _stance_suffix(stance_key)

    json_export = {
        'metadata': {
            'created': datetime.now().isoformat(),
            'project_name': Config.PROJECT_NAME,
            'research_question': Config.RESEARCH_QUESTION,
            'data_type': Config.DATA_TYPE,
            'stance': {
                'key': stance_key,
                'name': stance_info.get('name', ''),
                'description': stance_info.get('description', ''),
            } if stance_key else None,
            'total_codes': len(codebook),
            'quality_score': assessment['validation']['quality_score'],
            'extraction_focus': Config.EXTRACTION_FOCUS,
            'model': Config.MODEL,
            'similarity_threshold': Config.SIMILARITY_THRESHOLD
        },
        'code_groups': assessment.get('groups', {}),
        'codes': {}
    }

    sorted_codes = sorted(codebook.items(), key=lambda x: x[1].frequency, reverse=True)
    for label, code in sorted_codes:
        json_export['codes'][label] = {
            'definition': code.definition,
            'extraction_type': code.extraction_type,
            'code_group': code.code_group,
            'stance': code.stance,
            'inclusion': code.inclusion_criteria,
            'exclusion': code.exclusion_criteria,
            'examples': [ex['text'] for ex in code.examples],
            'frequency': code.frequency,
            'sources': code.source_documents
        }

    filepath = f"{Config.OUTPUT_PATH}codebook{suffix}.json"
    with open(filepath, 'w') as f:
        json.dump(json_export, f, indent=2)
    print(f"  codebook{suffix}.json")


def export_to_atlas_ti(codebook: Dict[str, CodeEntry], stance_key: str = ""):
    """Export in ATLAS.ti Excel format.

    ATLAS.ti imports Excel codebooks with columns by position:
      1: Code name (hierarchy via :: prefix, e.g., "Group::Subcode")
      2: Code comment/description
      3+: Code group(s)

    Import via: Import & Export tab > Codebook > Import from Excel
    """
    all_stances = get_all_stances()
    stance_name = all_stances.get(stance_key, {}).get('name', '') if stance_key else ''
    suffix = _stance_suffix(stance_key)

    wb = Workbook()
    ws = wb.active
    ws.title = "Codebook"

    # ATLAS.ti reads columns by position: Name, Comment, Group1, Group2...
    headers = ['Code', 'Comment', 'Code Group']
    ws.append(headers)

    header_fill = PatternFill(start_color="274C77", end_color="274C77", fill_type="solid")
    header_font = Font(color="FFFFFF", bold=True)
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font

    # Group codes by code_group for hierarchy
    groups = {}
    for label, code in codebook.items():
        group = code.code_group if code.code_group else 'General'
        if group not in groups:
            groups[group] = []
        groups[group].append((label, code))

    sorted_codes = sorted(codebook.items(), key=lambda x: x[1].frequency, reverse=True)
    for label, code in sorted_codes:
        group = code.code_group if code.code_group else 'General'
        # ATLAS.ti uses :: prefix for hierarchy (parent::child)
        code_name = f"{group}::{label.replace('_', ' ').title()}"

        # Build description with definition + inclusion/exclusion
        description = code.definition
        if code.inclusion_criteria:
            description += f"\n\nWhen to use: {'; '.join(code.inclusion_criteria)}"
        if code.exclusion_criteria:
            description += f"\nWhen NOT to use: {'; '.join(code.exclusion_criteria)}"

        row = [
            code_name,
            description,
            group,
        ]
        ws.append(row)

    ws.column_dimensions['A'].width = 40
    ws.column_dimensions['B'].width = 60
    ws.column_dimensions['C'].width = 25

    filepath = f"{Config.OUTPUT_PATH}codebook_atlasti{suffix}.xlsx"
    wb.save(filepath)
    print(f"  codebook_atlasti{suffix}.xlsx (Import via Import & Export > Codebook > Import from Excel)")


def export_to_nvivo_qdc(codebook: Dict[str, CodeEntry], stance_key: str = ""):
    """Export in REFI-QDA Codebook (.qdc) format for NVivo.

    QDC is the codebook-only exchange format (namespace urn:QDA-XML:codebook:1.0).
    NVivo imports via: Import > Codebook > select .qdc file.

    The file is plain XML rather than a zipped project archive.
    """
    all_stances = get_all_stances()
    stance_name = all_stances.get(stance_key, {}).get('name', 'Default') if stance_key else 'Default'
    suffix = _stance_suffix(stance_key)

    # Build QDC XML (REFI-QDA Codebook schema)
    codebook_root = ET.Element('CodeBook')
    codebook_root.set('xmlns', 'urn:QDA-XML:codebook:1.0')
    codebook_root.set('origin', f'AI Anthropology Toolkit — {stance_name}')

    codes_elem = ET.SubElement(codebook_root, 'Codes')

    # Group codes by code_group for hierarchy
    type_groups = {}
    for label, code in codebook.items():
        group = code.code_group if code.code_group else 'General'
        if group not in type_groups:
            type_groups[group] = []
        type_groups[group].append((label, code))

    for group_name, group_codes in sorted(type_groups.items()):
        # Parent code for the group (isCodable=false = container only)
        group_elem = ET.SubElement(codes_elem, 'Code')
        group_elem.set('guid', str(uuid.uuid4()))
        group_elem.set('name', group_name)
        group_elem.set('isCodable', 'false')

        sorted_group = sorted(group_codes, key=lambda x: x[1].frequency, reverse=True)
        for label, code in sorted_group:
            code_elem = ET.SubElement(group_elem, 'Code')
            code_elem.set('guid', str(uuid.uuid4()))
            code_elem.set('name', label.replace('_', ' ').title())
            code_elem.set('isCodable', 'true')

            # Description = definition + criteria
            desc_parts = [code.definition]
            if code.inclusion_criteria:
                desc_parts.append(f"When to use: {'; '.join(code.inclusion_criteria)}")
            if code.exclusion_criteria:
                desc_parts.append(f"When NOT to use: {'; '.join(code.exclusion_criteria)}")
            desc_elem = ET.SubElement(code_elem, 'Description')
            desc_elem.text = '\n'.join(desc_parts)

    # Pretty-print XML
    xml_str = minidom.parseString(ET.tostring(codebook_root, encoding='unicode')).toprettyxml(indent='  ')

    # QDC is a plain XML file (not zipped)
    qdc_path = f"{Config.OUTPUT_PATH}codebook_nvivo{suffix}.qdc"
    with open(qdc_path, 'w', encoding='utf-8') as f:
        f.write(xml_str)

    print(f"  codebook_nvivo{suffix}.qdc (Import via Import > Codebook)")

    export_nvivo_manual_guide(codebook, stance_key)


def export_nvivo_manual_guide(codebook: Dict[str, CodeEntry], stance_key: str = ""):
    """Create formatted guide for manual NVivo import with stance metadata"""
    all_stances = get_all_stances()
    stance_name = all_stances.get(stance_key, {}).get('name', 'Default') if stance_key else 'Default'
    suffix = _stance_suffix(stance_key)

    md_content = f"""# NVivo Codebook Import Guide — {stance_name}

## How to Import This Codebook into NVivo

**Option 1: QDC Import (Recommended)**
1. In NVivo, go to **Import > Codebook**
2. Select `codebook_nvivo{suffix}.qdc`
3. Click Open — codes are created with their hierarchy and descriptions

**Option 2: Manual Creation**
If QDC import doesn't preserve all details:
1. In NVivo, create each code via **Create > Code**, using the code names below
2. Paste the definition and criteria into the code's Description field (Code Properties)
3. Nest codes under parent codes named after each group to recreate the hierarchy

---

## Codes by Group

"""

    groups = {}
    for label, code in codebook.items():
        group = code.code_group if code.code_group else 'General'
        if group not in groups:
            groups[group] = []
        groups[group].append((label, code))

    for group_name in sorted(groups.keys()):
        md_content += f"### {group_name}\n\n"
        sorted_codes = sorted(groups[group_name], key=lambda x: x[1].frequency, reverse=True)
        for label, code in sorted_codes:
            md_content += f"**{label.replace('_', ' ').title()}** (freq: {code.frequency})\n\n"
            md_content += f"*Definition:* {code.definition}\n\n"
            if code.inclusion_criteria:
                md_content += f"*When to use:* {'; '.join(code.inclusion_criteria)}\n\n"
            if code.exclusion_criteria:
                md_content += f"*When NOT to use:* {'; '.join(code.exclusion_criteria)}\n\n"
            md_content += "---\n\n"

    filepath = f"{Config.OUTPUT_PATH}codebook_nvivo_guide{suffix}.md"
    with open(filepath, 'w') as f:
        f.write(md_content)
    print(f"  codebook_nvivo_guide{suffix}.md (Manual NVivo import guide)")



def export_to_markdown(codebook: Dict[str, CodeEntry], assessment: Dict, stance_key: str = ""):
    """Generate human-readable markdown documentation with stance context"""
    all_stances = get_all_stances()
    stance_info = all_stances.get(stance_key, {}) if stance_key else {}
    stance_name = stance_info.get('name', 'Default')
    suffix = _stance_suffix(stance_key)

    focus_names = {
        'theoretical': 'Theoretical Constructs',
        'methodological': 'Methodological Approaches',
        'emergent': 'Emergent Concepts'
    }
    focus_display = ', '.join([focus_names.get(f, f) for f in Config.EXTRACTION_FOCUS])

    md_content = f"""# Codebook Documentation

**Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M')}
**Project**: {Config.PROJECT_NAME if Config.PROJECT_NAME else 'Not specified'}
**Research Question**: {Config.RESEARCH_QUESTION if Config.RESEARCH_QUESTION else 'Not specified'}
**Data Type**: {DATA_TYPE_DESCRIPTIONS.get(Config.DATA_TYPE, Config.DATA_TYPE)}
**Analytical Lens**: {stance_name}
{"**Lens Description**: " + stance_info.get('description', '') if stance_info.get('description') else ''}
**Extraction Focus**: {focus_display}
**Quality Score**: {assessment['validation']['quality_score']:.2f}
**Model**: {Config.MODEL}

## Overview

Total Codes: {len(codebook)}

"""

    type_counts = Counter(code.extraction_type for code in codebook.values())
    md_content += "### Codes by Type\n\n"
    for ext_type, count in sorted(type_counts.items()):
        md_content += f"- {ext_type.title()}: {count}\n"
    md_content += "\n"

    group_counts = Counter(code.code_group for code in codebook.values())
    md_content += "### Codes by Group\n\n"
    for group, count in sorted(group_counts.items(), key=lambda x: x[1], reverse=True):
        md_content += f"- {group}: {count}\n"
    md_content += "\n"

    md_content += "## Code Definitions\n\n"

    groups = {}
    for label, code in codebook.items():
        group = code.code_group if code.code_group else 'Uncategorized'
        if group not in groups:
            groups[group] = []
        groups[group].append((label, code))

    for group_name in sorted(groups.keys()):
        md_content += f"### {group_name}\n\n"
        sorted_codes = sorted(groups[group_name], key=lambda x: x[1].frequency, reverse=True)

        for label, code in sorted_codes:
            md_content += f"#### {label} (frequency: {code.frequency})\n\n"
            md_content += f"**Definition**: {code.definition}\n\n"
            md_content += f"**Type**: {code.extraction_type.title() if code.extraction_type else 'Emergent'}\n\n"

            if code.inclusion_criteria:
                md_content += "**When to use**:\n"
                for criterion in code.inclusion_criteria:
                    md_content += f"- {criterion}\n"
                md_content += "\n"

            if code.exclusion_criteria:
                md_content += "**When NOT to use**:\n"
                for criterion in code.exclusion_criteria:
                    md_content += f"- {criterion}\n"
                md_content += "\n"

            if code.examples:
                md_content += "**Examples**:\n"
                for i, ex in enumerate(code.examples):
                    md_content += f"- \"{ex['text'][:200]}{'...' if len(ex['text']) > 200 else ''}\"\n"
                md_content += "\n"

            md_content += "---\n\n"

    filepath = f"{Config.OUTPUT_PATH}codebook_documentation{suffix}.md"
    with open(filepath, 'w') as f:
        f.write(md_content)
    print(f"  codebook_documentation{suffix}.md")


def export_all_formats(codebook: Dict[str, CodeEntry], assessment: Dict, stance_key: str = ""):
    """Export codebook in all supported formats for a single lens"""
    all_stances = get_all_stances()
    stance_name = all_stances.get(stance_key, {}).get('name', 'Default') if stance_key else 'Default'

    print(f"\n  Exporting [{stance_name}] codebook...")

    export_to_csv(codebook, stance_key)
    export_to_json(codebook, assessment, stance_key)
    export_to_atlas_ti(codebook, stance_key)
    export_to_nvivo_qdc(codebook, stance_key)
    export_to_markdown(codebook, assessment, stance_key)

    # Remove checkpoint file after successful completion
    suffix = _stance_suffix(stance_key)
    checkpoint_path = f"{Config.OUTPUT_PATH}codebook_in_progress{suffix}.json"
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)


print("✓ Export functions loaded (ATLAS.ti Excel, NVivo QDC, CSV, JSON, Markdown)")

## Processing Pipeline

*Orchestrate the complete codebook development workflow. When multiple lenses are selected, extraction runs in parallel—each lens produces its own codebook, refined and exported independently. The pipeline coordinates research context, lens-aware extraction, semantic deduplication, AI consolidation, and multi-format export.*

In [ ]:
# Main processing pipeline with parallel multi-lens support

def generate_codebook_for_stance(stance_key: str, documents: Dict[str, str],
                                  extraction_focus: List[str]) -> Tuple[str, Dict[str, CodeEntry], Dict, List]:
    """
    Generate a complete codebook for a single stance.
    Returns: (stance_key, codebook, assessment, extraction_log)
    """
    # Extract
    codebook, extraction_log = extract_codes_batch(documents, stance_key, extraction_focus)

    if not codebook:
        return (stance_key, {}, {}, extraction_log)

    # Refine
    codebook, assessment = refine_and_assess(codebook, stance_key)

    return (stance_key, codebook, assessment, extraction_log)


def develop_codebook_pipeline():
    """
    Main pipeline for codebook development.
    Supports parallel multi-stance codebook generation.
    Each stance produces its own codebook, refined and exported independently.
    """
    print("=" * 60)
    print("CODEBOOK DEVELOPMENT PIPELINE")
    print("=" * 60)

    # Validate prerequisites
    if not uploaded_documents:
        print("\nNo documents uploaded. Please upload documents first.")
        return None, None

    if not client:
        print("\nAPI not configured. Please apply configuration first.")
        return None, None

    if not Config.EXTRACTION_FOCUS:
        print("\nNo extraction focus selected. Please select at least one focus area.")
        return None, None

    if not Config.SELECTED_STANCES:
        print("\nNo analytical lenses selected. Please select at least one lens.")
        return None, None

    # Display configuration summary
    all_stances = get_all_stances()
    stance_names = [all_stances[s]['name'] for s in Config.SELECTED_STANCES]
    focus_names = {'theoretical': 'Theoretical', 'methodological': 'Methodological', 'emergent': 'Emergent'}
    focus_display = ', '.join([focus_names[f] for f in Config.EXTRACTION_FOCUS])

    total_words = sum(len(doc.split()) for doc in uploaded_documents.values())

    print(f"\n  Project: {Config.PROJECT_NAME or '(not set)'}")
    if Config.RESEARCH_QUESTION:
        print(f"  Research Question: {Config.RESEARCH_QUESTION[:100]}{'...' if len(Config.RESEARCH_QUESTION) > 100 else ''}")
    print(f"  Data Type: {DATA_TYPE_DESCRIPTIONS.get(Config.DATA_TYPE, Config.DATA_TYPE)}")
    print(f"  Documents: {len(uploaded_documents)} ({total_words:,} words)")
    print(f"  Model: {Config.MODEL}")
    print(f"  Extraction Focus: {focus_display}")
    print(f"  Analytical Lenses: {', '.join(stance_names)}")
    print(f"  Max Codes: {Config.MAX_TOTAL_CODES} per lens")
    print(f"  Similarity Threshold: {Config.SIMILARITY_THRESHOLD:.2f}")

    start_time = time.time()
    all_results = {}  # stance_key -> (codebook, assessment, extraction_log)

    # ── Multi-Lens Generation ────────────────────────────────────────

    if len(Config.SELECTED_STANCES) > 1:
        print(f"\n{'─' * 60}")
        print(f"Parallel Multi-Lens Generation ({len(Config.SELECTED_STANCES)} lenses)")
        max_workers = min(3, len(Config.SELECTED_STANCES))
        print(f"  Processing up to {max_workers} lenses concurrently (combined API concurrency capped near 9 calls)")
        print(f"{'─' * 60}")

        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(
                    generate_codebook_for_stance,
                    stance_key,
                    uploaded_documents,
                    Config.EXTRACTION_FOCUS
                ): stance_key
                for stance_key in Config.SELECTED_STANCES
            }

            for future in concurrent.futures.as_completed(futures):
                stance_key = futures[future]
                try:
                    stance_key_result, codebook, assessment, log = future.result()
                    all_results[stance_key_result] = (codebook, assessment, log)
                except Exception as e:
                    print(f"\n  [{all_stances[stance_key]['name']}] Failed: {e}")
                    all_results[stance_key] = ({}, {}, [])

    else:
        # Single lens — run directly, with the same failure isolation as the multi-lens path
        stance_key = Config.SELECTED_STANCES[0]
        print(f"\n{'─' * 60}")
        print(f"Single-Lens Generation: {all_stances[stance_key]['name']}")
        print(f"{'─' * 60}")

        try:
            stance_key_result, codebook, assessment, log = generate_codebook_for_stance(
                stance_key, uploaded_documents, Config.EXTRACTION_FOCUS)
            all_results[stance_key_result] = (codebook, assessment, log)
        except Exception as e:
            print(f"\n  [{all_stances[stance_key]['name']}] Failed: {e}")
            all_results[stance_key] = ({}, {}, [])

    extraction_time = time.time() - start_time

    # ── Export ─────────────────────────────────────────────────────────

    print(f"\n{'─' * 60}")
    print(f"Exporting Codebooks")
    print(f"{'─' * 60}")

    valid_results = {}
    for stance_key, (codebook, assessment, log) in all_results.items():
        if codebook:
            export_all_formats(codebook, assessment, stance_key)
            valid_results[stance_key] = (codebook, assessment, log)
        else:
            print(f"\n  [{all_stances[stance_key]['name']}] No codes extracted — skipping export")

    # ── Summary ────────────────────────────────────────────────────────

    print(f"\n{'=' * 60}")
    print(f"CODEBOOK DEVELOPMENT COMPLETE")
    print(f"{'=' * 60}")
    print(f"\n  Processing time: {extraction_time:.1f}s")
    print(f"  Lenses completed: {len(valid_results)}/{len(Config.SELECTED_STANCES)}")

    for stance_key, (codebook, assessment, log) in valid_results.items():
        stance_name = all_stances[stance_key]['name']
        type_counts = Counter(code.extraction_type for code in codebook.values())
        print(f"\n  [{stance_name}]")
        print(f"    Total codes: {len(codebook)}")
        print(f"    Quality score: {assessment['validation']['quality_score']:.2f}")
        print(f"    By type: {dict(type_counts)}")

    print(f"\n  Output directory: {Config.OUTPUT_PATH}")
    print(f"\n  Import instructions:")
    print(f"    ATLAS.ti: Import & Export > Codebook > Import from Excel (.xlsx)")
    print(f"    NVivo: Import > Codebook (.qdc)")
    print(f"    NVivo Guide: See codebook_nvivo_guide_[lens].md for manual import steps")

    # Return results — for single stance, return the codebook directly for backwards compat
    if len(valid_results) == 1:
        stance_key = list(valid_results.keys())[0]
        return valid_results[stance_key][0], valid_results[stance_key][1]
    else:
        return valid_results, None


print("✓ Multi-lens pipeline ready (up to 3 lenses in parallel, combined API concurrency capped near 9 calls)")

## Execute Codebook Development

*Run the complete codebook development process with your uploaded documents and configured parameters. If multiple lenses are selected, codebooks are generated in parallel. Monitor progress through each development stage and generate final outputs ready for research application.*

In [ ]:
# Execute the pipeline
results = develop_codebook_pipeline()

## Output Files

*View the generated codebook files. Each lens produces its own set of export files (CSV, JSON, ATLAS.ti Excel, NVivo QDC, NVivo manual guide, Markdown). Access them from the Colab file browser on the left side, then right-click to download.*

In [ ]:
# Output file listing

def show_output_files():
    """Display list of generated output files"""

    output_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>Output Files</h3>
    <p>Your codebook(s) have been exported in multiple formats. Each lens generates its own set of files.</p>
    <p style='font-family: monospace; background-color: #A3CEF1; padding: 10px; border-radius: 5px;'>/content/codebook_outputs/</p>
    <ul style='color: #274C77;'>
        <li><strong>ATLAS.ti</strong>: codebook_atlasti_[lens].xlsx — Import via Codes > Import > Codebook</li>
        <li><strong>NVivo</strong>: codebook_nvivo_[lens].qdc — REFI-QDA codebook (QDC); import via the Import ribbon > Codebook</li>
        <li><strong>NVivo Guide</strong>: codebook_nvivo_guide_[lens].md — Manual import guide if QDC import doesn't work</li>
        <li><strong>General</strong>: codebook_[lens].csv, codebook_[lens].json</li>
        <li><strong>Documentation</strong>: codebook_documentation_[lens].md</li>
    </ul>
    <p style='color: #6096BA; font-size: 13px;'>Right-click any file in the file browser and select "Download" to save to your computer.</p>
    </div>
    """

    display(HTML(output_html))

    # List actual files
    print("Generated files:")
    if os.path.exists(Config.OUTPUT_PATH):
        for f in sorted(os.listdir(Config.OUTPUT_PATH)):
            filepath = os.path.join(Config.OUTPUT_PATH, f)
            size = os.path.getsize(filepath)
            if size > 1024:
                size_str = f"{size/1024:.1f} KB"
            else:
                size_str = f"{size} bytes"
            print(f"   {f} ({size_str})")
    else:
        print("   No output directory found. Run the pipeline first.")

show_output_files()